In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score



In [ ]:
from google.colab import files
df = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import io
import pandas as pd

# Assuming `df` is currently the dictionary from files.upload()
if isinstance(df, dict):
    # Extract the filename from the dictionary keys
    excel_filenames = [k for k in df.keys() if k.endswith('.xlsx') or k.endswith('.xls')]
    if excel_filenames:
        filename = excel_filenames[0]
        file_content_bytes = df[filename] # This should be bytes
        excel_file_object = io.BytesIO(file_content_bytes)
        df = pd.read_excel(excel_file_object) # Overwrite df with the DataFrame
        print(f"Excel file '{filename}' loaded successfully into DataFrame.")
        print("First 5 rows of the new DataFrame:")
        print(df.head())
    else:
        print("Error: 'df' is a dictionary but no Excel file found in its keys.")
else:
    print("df is already a DataFrame, no conversion needed.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io # Added for BytesIO

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# --- Start of fix for df being a dictionary ---
# Assuming `df` is currently the dictionary from files.upload()
if isinstance(df, dict):
    # Extract the filename from the dictionary keys
    # Check if 'df' is indeed from files.upload() by checking its keys
    excel_filenames = [k for k in df.keys() if k.endswith('.xlsx') or k.endswith('.xls')]
    if excel_filenames:
        filename = excel_filenames[0]
        file_content_bytes = df[filename] # This should be bytes
        excel_file_object = io.BytesIO(file_content_bytes)
        df = pd.read_excel(excel_file_object) # Overwrite df with the DataFrame
        print(f"Excel file '{filename}' loaded successfully into DataFrame.")
    else:
        print("Warning: 'df' is a dictionary but no Excel file found in its keys. Analysis might fail.")
        # If df is a dict but not holding excel data, the next lines will likely fail.
        # This case might not be expected if previous cells ran correctly.
else:
    print("df is already a DataFrame, proceeding with analysis.")

# --- End of fix for df being a dictionary ---

# Now, re-derive 'attempts' and 'columns' from the correctly loaded DataFrame `df`
# This will fix any previous incorrect state or type of 'attempts' or 'columns'
attempts = df.iloc[:, 0].values.reshape(-1, 1)

# Get all column names except the first one initially
all_other_columns = df.columns[1:]

# Filter for only numeric columns for analysis
columns = []
for col_name in all_other_columns:
    if pd.api.types.is_numeric_dtype(df[col_name]):
        columns.append(col_name)

print("Identified experimental columns for analysis:", columns)


# =======================================
# 2. Набор тестируемых функций (Re-defined for self-contained execution)
# =======================================

def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None), "poly2": (-np.inf, None), "poly3": (-np.inf, None),
            "log": (-np.inf, None), "exp": (-np.inf, None), "power": (-np.inf, None),
            "root": (-np.inf, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X))

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)))

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)))

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)))
    else:
        results["log"] = (-np.inf, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
        )
    else:
        results["exp"] = (-np.inf, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X)))
        )
    else:
        results["power"] = (-np.inf, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)))
    else:
        results["root"] = (-np.inf, None)

    return results


# =======================================
# 3. Метод 1: RANSAC + Polynomial/Linear (Re-defined for self-contained execution, with robustness improvements)
# =======================================

def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan) # Return empty inliers and a prediction function that returns NaN

    poly = PolynomialFeatures(degree=2)  # up to quadratic features
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            # Need enough samples for PolynomialFeatures to transform and then for LinearRegression
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                # Not enough samples for even simple polyfit after transform, return NaN predictions
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails
        return inliers_mask, predict_func

    try:
        # Add random_state for reproducibility
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        # If RANSAC fails, fall back to simple linear regression
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails

    return inliers_mask, predict_func


print("Starting RANSAC and Model Fitting for each experimental column...")

for col in columns:
    y_raw = df[col].values.reshape(-1, 1)

    # Combine attempts and y_raw, then drop rows with NaNs
    temp_data = pd.DataFrame({'attempts_val': attempts.flatten(), 'y_val': y_raw.flatten()})
    cleaned_data = temp_data.dropna()

    # Check if there is enough data after cleaning
    if cleaned_data.shape[0] < 2: # Need at least 2 points for linear regression, more for polynomial or RANSAC
        print(f"Skipping column '{col}' due to insufficient non-NaN data after cleaning (less than 2 data points).")
        continue

    x_cleaned = cleaned_data['attempts_val'].values.reshape(-1, 1)
    y_cleaned = cleaned_data['y_val'].values.reshape(-1, 1)

    # ---- A. Метод RANSAC ----
    inliers, ransac_predict = None, None
    try:
        inliers, ransac_predict = ransac_fit(x_cleaned, y_cleaned)
    except Exception as e:
        print(f"Critical Error: ransac_fit function call failed for column '{col}'. Error: {e}. Skipping RANSAC plot for this column.")
        inliers = np.ones(len(x_cleaned), dtype=bool) # Assume all are inliers if RANSAC fails critically
        ransac_predict = lambda X: np.full_like(X, np.nan)


    # ---- B. Перебор функций ----
    models = try_all_models(x_cleaned, y_cleaned)
    # Filter models with valid prediction functions and R2 scores that are not -inf
    valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

    best_type = None
    best_r2 = -np.inf
    best_predict = None

    if valid_models:
        best_type = max(valid_models, key=lambda k: valid_models[k][0])
        best_r2, best_predict = valid_models[best_type]
    else:
        print(f"Skipping column '{col}' as no valid mathematical models could be fitted after cleaning data.")
        continue # Skip plotting if no models can be fitted


    # ---- Визуализация ----
    plt.figure(figsize=(12, 7))
    x_plot_min = x_cleaned.min()
    x_plot_max = x_cleaned.max()

    # Handle cases where x_cleaned has only one unique value (or min == max)
    if x_plot_min == x_plot_max:
        # If all x values are the same, can't create a range for plotting a curve.
        # Just plot points, no curve for continuous function.
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)


    # Plot raw data
    plt.scatter(x_cleaned, y_cleaned, s=40, label="Сырые данные", alpha=0.7, color='skyblue')

    # Plot RANSAC inliers
    if inliers is not None and len(inliers) == len(x_cleaned) and np.any(inliers):
        plt.scatter(x_cleaned[inliers], y_cleaned[inliers], s=70, label="RANSAC — без выбросов", facecolors='none', edgecolors='green', linewidth=2)
    elif inliers is not None and not np.any(inliers):
        print(f"No inliers identified by RANSAC for column '{col}'.")


    # Plot RANSAC curve
    if ransac_predict is not None and x_plot.size > 0:
        try:
            ransac_predictions = ransac_predict(x_plot)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot, ransac_predictions, linewidth=2.5, color='orange', label="RANSAC модель")
            else:
                print(f"RANSAC model predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating RANSAC predictions for column '{col}': {e}. Not plotting RANSAC curve.")


    # Plot best mathematical function
    if best_predict is not None and x_plot.size > 0:
        try:
            best_model_predictions = best_predict(x_plot)
            if not np.all(np.isnan(best_model_predictions)):
                plt.plot(x_plot, best_model_predictions, linewidth=3, color='red', linestyle='--', label=f"Лучшая функция: {best_type} (R²={best_r2:.3f})")
            else:
                print(f"Best mathematical function predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating best model predictions for column '{col}': {e}. Not plotting best model curve.")


    plt.title(f"Столбец: {col}", fontsize=16)
    plt.xlabel("Номер попытки", fontsize=12)
    plt.ylabel("Значение разности потенциалов", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("\nAnalysis and plotting complete for all identified experimental columns.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io # Added for BytesIO

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# --- Start of fix for df being a dictionary ---
# Assuming `df` is currently the dictionary from files.upload()
if isinstance(df, dict):
    # Extract the filename from the dictionary keys
    # Check if 'df' is indeed from files.upload() by checking its keys
    excel_filenames = [k for k in df.keys() if k.endswith('.xlsx') or k.endswith('.xls')]
    if excel_filenames:
        filename = excel_filenames[0]
        file_content_bytes = df[filename] # This should be bytes
        excel_file_object = io.BytesIO(file_content_bytes)
        df = pd.read_excel(excel_file_object) # Overwrite df with the DataFrame
        print(f"Excel file '{filename}' loaded successfully into DataFrame.")
    else:
        print("Warning: 'df' is a dictionary but no Excel file found in its keys. Analysis might fail.")
        # If df is a dict but not holding excel data, the next lines will likely fail.
        # This case might not be expected if previous cells ran correctly.
else:
    print("df is already a DataFrame, proceeding with analysis.")

# --- End of fix for df being a dictionary ---

# Now, re-derive 'attempts' and 'columns' from the correctly loaded DataFrame `df`
# This will fix any previous incorrect state or type of 'attempts' or 'columns'
attempts = df.iloc[:, 0].values.reshape(-1, 1)

# Get all column names except the first one initially
all_other_columns = df.columns[1:]

# Filter for only numeric columns for analysis
columns = []
for col_name in all_other_columns:
    if pd.api.types.is_numeric_dtype(df[col_name]):
        columns.append(col_name)

print("Identified experimental columns for analysis:", columns)


# =======================================
# 2. Набор тестируемых функций (Re-defined for self-contained execution)
# =======================================

def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None), "poly2": (-np.inf, None), "poly3": (-np.inf, None),
            "log": (-np.inf, None), "exp": (-np.inf, None), "power": (-np.inf, None),
            "root": (-np.inf, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X))

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)))

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)))

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)))
    else:
        results["log"] = (-np.inf, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
        )
    else:
        results["exp"] = (-np.inf, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X)))
        )
    else:
        results["power"] = (-np.inf, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)))
    else:
        results["root"] = (-np.inf, None)

    return results


# =======================================
# 3. Метод 1: RANSAC + Polynomial/Linear (Re-defined for self-contained execution, with robustness improvements)
# =======================================

def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan) # Return empty inliers and a prediction function that returns NaN

    poly = PolynomialFeatures(degree=2)  # up to quadratic features
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            # Need enough samples for PolynomialFeatures to transform and then for LinearRegression
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                # Not enough samples for even simple polyfit after transform, return NaN predictions
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails
        return inliers_mask, predict_func

    try:
        # Add random_state for reproducibility
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        # If RANSAC fails, fall back to simple linear regression
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails

    return inliers_mask, predict_func


print("Starting RANSAC and Model Fitting for each experimental column...")

for col in columns:
    y_raw = df[col].values.reshape(-1, 1)

    # Combine attempts and y_raw, then drop rows with NaNs
    temp_data = pd.DataFrame({'attempts_val': attempts.flatten(), 'y_val': y_raw.flatten()})
    cleaned_data = temp_data.dropna()

    # Check if there is enough data after cleaning
    if cleaned_data.shape[0] < 2: # Need at least 2 points for linear regression, more for polynomial or RANSAC
        print(f"Skipping column '{col}' due to insufficient non-NaN data after cleaning (less than 2 data points).")
        continue

    x_cleaned = cleaned_data['attempts_val'].values.reshape(-1, 1)
    y_cleaned = cleaned_data['y_val'].values.reshape(-1, 1)

    # ---- A. Метод RANSAC ----
    inliers, ransac_predict = None, None
    try:
        inliers, ransac_predict = ransac_fit(x_cleaned, y_cleaned)
    except Exception as e:
        print(f"Critical Error: ransac_fit function call failed for column '{col}'. Error: {e}. Skipping RANSAC plot for this column.")
        inliers = np.ones(len(x_cleaned), dtype=bool) # Assume all are inliers if RANSAC fails critically
        ransac_predict = lambda X: np.full_like(X, np.nan)


    # ---- B. Перебор функций ----
    models = try_all_models(x_cleaned, y_cleaned)
    # Filter models with valid prediction functions and R2 scores that are not -inf
    valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

    best_type = None
    best_r2 = -np.inf
    best_predict = None

    if valid_models:
        best_type = max(valid_models, key=lambda k: valid_models[k][0])
        best_r2, best_predict = valid_models[best_type]
    else:
        print(f"Skipping column '{col}' as no valid mathematical models could be fitted after cleaning data.")
        continue # Skip plotting if no models can be fitted


    # ---- Визуализация ----
    plt.figure(figsize=(12, 7))
    x_plot_min = x_cleaned.min()
    x_plot_max = x_cleaned.max()

    # Handle cases where x_cleaned has only one unique value (or min == max)
    if x_plot_min == x_plot_max:
        # If all x values are the same, can't create a range for plotting a curve.
        # Just plot points, no curve for continuous function.
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)


    # Plot raw data
    plt.scatter(x_cleaned, y_cleaned, s=40, label="Сырые данные", alpha=0.7, color='skyblue')

    # Plot RANSAC inliers
    if inliers is not None and len(inliers) == len(x_cleaned) and np.any(inliers):
        plt.scatter(x_cleaned[inliers], y_cleaned[inliers], s=70, label="RANSAC — без выбросов", facecolors='none', edgecolors='green', linewidth=2)
    elif inliers is not None and not np.any(inliers):
        print(f"No inliers identified by RANSAC for column '{col}'.")


    # Plot RANSAC curve
    if ransac_predict is not None and x_plot.size > 0:
        try:
            ransac_predictions = ransac_predict(x_plot)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot, ransac_predictions, linewidth=2.5, color='orange', label="RANSAC модель")
            else:
                print(f"RANSAC model predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating RANSAC predictions for column '{col}': {e}. Not plotting RANSAC curve.")


    # Plot best mathematical function
    if best_predict is not None and x_plot.size > 0:
        try:
            best_model_predictions = best_predict(x_plot)
            if not np.all(np.isnan(best_model_predictions)):
                plt.plot(x_plot, best_model_predictions, linewidth=3, color='red', linestyle='--', label=f"Лучшая функция: {best_type} (R²={best_r2:.3f})")
            else:
                print(f"Best mathematical function predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating best model predictions for column '{col}': {e}. Not plotting best model curve.")


    plt.title(f"Столбец: {col}", fontsize=16)
    plt.xlabel("Номер попытки", fontsize=12)
    plt.ylabel("Значение", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("\nAnalysis and plotting complete for all identified experimental columns.")

In [ ]:
import re

# 1. Extract the first column as the independent variable `attempts`
attempts = df.iloc[:, 0].values.reshape(-1, 1)

# Prepare a list to store aggregated columns for the new DataFrame
aggregated_data = pd.DataFrame(attempts, columns=[df.columns[0]])

# Define specific series for Nitril groups (using exact column names from df)
nitril_group_1_3_series = ['нитрил (серия 1)', 'нитрил (серия 2)', 'нитрил (серия 3)']
nitril_group_4_6_series = ['нитрил (серия 4)', 'нитрил (серия 5)', 'Нитрил (серия 6)']

# Initialize material groups for collecting columns, now including specific ones for Nitril
material_groups_to_process = {
    'хлопок': [],
    'полиэстер': [],
    'шерсть кота': [],
    'кожа': [],
    'нитрил (серии 1-3)': [], # New group for nitril 1-3
    'нитрил (серии 4-6)': []  # New group for nitril 4-6
}

# Iterate through columns (excluding the first one which is 'attempts')
for col_name in df.columns[1:]:
    if pd.api.types.is_numeric_dtype(df[col_name]):
        # Assign to specific nitril groups if applicable
        if col_name in nitril_group_1_3_series:
            material_groups_to_process['нитрил (серии 1-3)'].append(col_name)
        elif col_name in nitril_group_4_6_series:
            material_groups_to_process['нитрил (серии 4-6)'].append(col_name)
        else: # For other materials (and 'нитрил (пробная выборка)' which is not in the above lists)
            normalized_col_name = col_name.lower()
            # General matching for other material prefixes
            for material_key_prefix in ['хлопок', 'полиэстер', 'шерсть кота', 'кожа']:
                if material_key_prefix in normalized_col_name:
                    material_groups_to_process[material_key_prefix].append(col_name)
                    break # Assuming a column belongs to only one group

# 4. Calculate the row-wise mean for each material group
new_columns_list = []
for material_key_name, cols_to_aggregate in material_groups_to_process.items():
    if cols_to_aggregate:
        # Capitalize the first letter of each word in the group name for display
        display_name = ' '.join([word.capitalize() for word in material_key_name.split(' ')])
        aggregated_col_name = f"{display_name} (среднее)"

        aggregated_data[aggregated_col_name] = df[cols_to_aggregate].mean(axis=1, skipna=True)
        new_columns_list.append(aggregated_col_name)
    else:
        print(f"No columns found for material group '{material_key_name}'.")

# 6. Update the `columns` variable
columns = new_columns_list

print("Aggregated DataFrame head:")
print(aggregated_data.head())
print("\nUpdated list of columns for analysis:", columns)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# =======================================
# 2. Набор тестируемых функций
# =======================================

def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None), "poly2": (-np.inf, None), "poly3": (-np.inf, None),
            "log": (-np.inf, None), "exp": (-np.inf, None), "power": (-np.inf, None),
            "root": (-np.inf, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X))

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)))

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)))

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)))
    else:
        results["log"] = (-np.inf, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
        )
    else:
        results["exp"] = (-np.inf, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X)))
        )
    else:
        results["power"] = (-np.inf, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)))
    else:
        results["root"] = (-np.inf, None)

    return results


# =======================================
# 3. Метод 1: RANSAC + Polynomial/Linear
# =======================================

def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan) # Return empty inliers and a prediction function that returns NaN

    poly = PolynomialFeatures(degree=2)  # up to quadratic features
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            # Need enough samples for PolynomialFeatures to transform and then for LinearRegression
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                # Not enough samples for even simple polyfit after transform, return NaN predictions
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails
        return inliers_mask, predict_func

    try:
        # Add random_state for reproducibility
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        # If RANSAC fails, fall back to simple linear regression
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass # Prediction function remains NaN if fallback also fails

    return inliers_mask, predict_func

print("Helper functions `try_all_models` and `ransac_fit` defined successfully.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# The variables `aggregated_data` and `columns` are assumed to be correctly defined
# from previous execution cells after the aggregation step.
# The helper functions `try_all_models` and `ransac_fit` are also assumed to be
# already defined from a previous cell and are robust.

print("Starting RANSAC and Model Fitting for each AGGREGATED experimental column...")

# Extract `attempts` from the aggregated_data DataFrame
attempts_agg = aggregated_data.iloc[:, 0].values.reshape(-1, 1)

for col in columns:
    y_raw = aggregated_data[col].values.reshape(-1, 1)

    # Combine attempts and y_raw, then drop rows with NaNs
    temp_data = pd.DataFrame({'attempts_val': attempts_agg.flatten(), 'y_val': y_raw.flatten()})
    cleaned_data = temp_data.dropna()

    # Check if there is enough data after cleaning
    if cleaned_data.shape[0] < 2: # Need at least 2 points for linear regression, more for polynomial or RANSAC
        print(f"Skipping aggregated column '{col}' due to insufficient non-NaN data after cleaning (less than 2 data points).")
        continue

    x_cleaned = cleaned_data['attempts_val'].values.reshape(-1, 1)
    y_cleaned = cleaned_data['y_val'].values.reshape(-1, 1)

    # ---- A. Метод RANSAC ----
    inliers, ransac_predict = None, None
    try:
        inliers, ransac_predict = ransac_fit(x_cleaned, y_cleaned)
    except Exception as e:
        print(f"Critical Error: ransac_fit function call failed for aggregated column '{col}'. Error: {e}. Skipping RANSAC plot for this column.")
        inliers = np.ones(len(x_cleaned), dtype=bool) # Assume all are inliers if RANSAC fails critically
        ransac_predict = lambda X: np.full_like(X, np.nan)


    # ---- B. Перебор функций ----
    models = try_all_models(x_cleaned, y_cleaned)
    # Filter models with valid prediction functions and R2 scores that are not -inf
    valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

    best_type = None
    best_r2 = -np.inf
    best_predict = None

    if valid_models:
        best_type = max(valid_models, key=lambda k: valid_models[k][0])
        best_r2, best_predict = valid_models[best_type]
    else:
        print(f"Skipping aggregated column '{col}' as no valid mathematical models could be fitted after cleaning data.")
        continue # Skip plotting if no models can be fitted


    # ---- Визуализация ----
    plt.figure(figsize=(12, 7))
    x_plot_min = x_cleaned.min()
    x_plot_max = x_cleaned.max()

    # Handle cases where x_cleaned has only one unique value (or min == max)
    if x_plot_min == x_plot_max:
        # If all x values are the same, can't create a range for plotting a curve.
        # Just plot points, no curve for continuous function.
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)


    # Plot raw data
    plt.scatter(x_cleaned, y_cleaned, s=40, label="Сырые данные", alpha=0.7, color='skyblue')

    # Plot RANSAC inliers
    if inliers is not None and len(inliers) == len(x_cleaned) and np.any(inliers):
        plt.scatter(x_cleaned[inliers], y_cleaned[inliers], s=70, label="RANSAC — без выбросов", facecolors='none', edgecolors='green', linewidth=2)
    elif inliers is not None and not np.any(inliers):
        print(f"No inliers identified by RANSAC for aggregated column '{col}'.")


    # Plot RANSAC curve
    if ransac_predict is not None and x_plot.size > 0:
        try:
            ransac_predictions = ransac_predict(x_plot)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot, ransac_predictions, linewidth=2.5, color='orange', label="RANSAC модель")
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot[np.argmin(ransac_predictions)][0]
                max_x_ransac = x_plot[np.argmax(ransac_predictions)][0]
                print(f"  RANSAC Model Extrema for '{col}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"RANSAC model predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating RANSAC predictions for aggregated column '{col}': {e}. Not plotting RANSAC curve.")


    # Plot best mathematical function
    if best_predict is not None and x_plot.size > 0:
        try:
            best_model_predictions = best_predict(x_plot)
            if not np.all(np.isnan(best_model_predictions)): # Only plot if predictions are not all NaNs
                model_description = f"Лучшая функция: {best_type} (R²={best_r2:.3f})"
                if best_type == 'linear':
                    model_description += " (Полином 1-й степени)"
                elif best_type == 'poly2':
                    model_description += " (Полином 2-й степени)"
                elif best_type == 'poly3':
                    model_description += " (Полином 3-й степени)"
                else:
                    model_description += " (Не полиномиальная функция)"

                plt.plot(x_plot, best_model_predictions, linewidth=3, color='red', linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot[np.argmin(best_model_predictions)][0]
                max_x_best = x_plot[np.argmax(best_model_predictions)][0]
                print(f"  Best Model Extrema for '{col}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
            else:
                print(f"Best mathematical function predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating best model predictions for aggregated column '{col}': {e}. Not plotting best model curve.")


    plt.title(f"Столбец: {col}", fontsize=16)
    plt.xlabel("Номер попытки", fontsize=12)
    plt.ylabel("Значение разности потенциалов", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("\nAnalysis and plotting complete for all identified AGGREGATED experimental columns.")

In [ ]:
import numpy as np
import pandas as pd
import re

# 1. Extract the first column as the independent variable `attempts_base`
attempts_base = df.iloc[:, 0].values.reshape(-1, 1)
max_attempts = attempts_base.max() if attempts_base.size > 0 else 0

# Dictionary to store scaled and concatenated data for each material
scaled_concatenated_data = {}

# Define specific series for Nitril groups (using exact column names from df)
nitril_group_1_3_series_f33 = ['нитрил (серия 1)', 'нитрил (серия 2)', 'нитрил (серия 3)']
nitril_group_4_6_series_f33 = ['нитрил (серия 4)', 'нитрил (серия 5)', 'Нитрил (серия 6)']

# Initialize material groups for collecting columns, now including specific ones for Nitril
material_groups_f33 = {
    'хлопок': [],
    'полиэстер': [],
    'шерсть кота': [],
    'кожа': [],
    'нитрил (серии 1-3)': [], # New group for nitril 1-3
    'нитрил (серии 4-6)': []  # New group for nitril 4-6
}

# Assign columns to groups
for col_name in df.columns[1:]:
    if pd.api.types.is_numeric_dtype(df[col_name]):
        # Assign to specific nitril groups if applicable
        if col_name in nitril_group_1_3_series_f33:
            material_groups_f33['нитрил (серии 1-3)'].append(col_name)
        elif col_name in nitril_group_4_6_series_f33:
            material_groups_f33['нитрил (серии 4-6)'].append(col_name)
        else: # For other materials (and 'нитрил (пробная выборка)')
            normalized_col_name = col_name.lower()
            # General matching for other material prefixes
            for material_key_prefix in ['хлопок', 'полиэстер', 'шерсть кота', 'кожа']:
                if material_key_prefix in normalized_col_name:
                    material_groups_f33[material_key_prefix].append(col_name)
                    break # Assuming a column belongs to only one group


# Process each material group: scale attempts and concatenate y values
for material_key_name, cols_to_aggregate in material_groups_f33.items(): # Use the new material_groups_f33
    if not cols_to_aggregate:
        print(f"No series found for material group '{material_key_name}'. Skipping.")
        continue

    all_x_scaled_for_material = []
    all_y_concatenated_for_material = []

    # Iterate through each series for the current material
    for series_idx, col_name in enumerate(cols_to_aggregate):
        # Scale 'attempts' for the current series
        current_attempts_scaled = attempts_base + series_idx * max_attempts
        current_y_values = df[col_name].values.reshape(-1, 1)

        # Combine x and y, then drop rows with NaNs for the current series
        temp_series_data = pd.DataFrame({
            'x_val': current_attempts_scaled.flatten(),
            'y_val': current_y_values.flatten()
        })
        cleaned_series_data = temp_series_data.dropna()

        if cleaned_series_data.shape[0] > 0:
            all_x_scaled_for_material.append(cleaned_series_data['x_val'].values.reshape(-1, 1))
            all_y_concatenated_for_material.append(cleaned_series_data['y_val'].values.reshape(-1, 1))

    # Concatenate all series data for the current material
    if all_x_scaled_for_material and all_y_concatenated_for_material:
        x_final = np.vstack(all_x_scaled_for_material)
        y_final = np.vstack(all_y_concatenated_for_material)

        # Store the processed data for the material
        # Capitalize the first letter of each word in the group name for display
        display_material_name = ' '.join([word.capitalize() for word in material_key_name.split(' ')])
        scaled_concatenated_data[f"{display_material_name} (с объединенными сериями)"] = {
            'x_scaled': x_final,
            'y_concatenated': y_final
        }
    else:
        print(f"Insufficient data after cleaning for material group '{material_key_name}'. Skipping.")

# Update the `columns` variable for further analysis
columns = list(scaled_concatenated_data.keys())

print("Prepared scaled and concatenated data for materials:")
for material_name, data in scaled_concatenated_data.items():
    print(f"  - {material_name}: x_scaled shape {data['x_scaled'].shape}, y_concatenated shape {data['y_concatenated'].shape}")
print("\nUpdated list of columns for analysis:", columns)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # Keep for plotting later

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.ensemble import IsolationForest # New import for Isolation Forest
from scipy.signal import savgol_filter # New import for Savitzky-Golay filter

# =======================================
# 1. Isolation Forest Filter
# =======================================
def isolation_forest_filter(x, y, contamination=0.1):
    # Combine x and y for Isolation Forest
    data_for_if = np.hstack((x, y))

    # Ensure there's enough data and features for IsolationForest
    if data_for_if.shape[0] < 2 or data_for_if.shape[1] < 2: # Min 2 samples, min 2 features (x and y)
        return np.ones(x.shape[0], dtype=bool) # If not enough data, assume all are inliers

    try:
        # Fit Isolation Forest model
        iso_forest = IsolationForest(random_state=42, contamination=contamination)
        outlier_preds = iso_forest.fit_predict(data_for_if)
        # -1 indicates an outlier, 1 indicates an inlier
        inlier_mask = (outlier_preds != -1)
    except Exception as e:
        print(f"Warning: Isolation Forest failed with error: {e}. All data points will be considered inliers.")
        inlier_mask = np.ones(x.shape[0], dtype=bool) # Fallback to all inliers if Isolation Forest fails

    return inlier_mask


# =======================================
# 2. RANSAC + Polynomial/Linear (re-defined for self-contained execution)
# =======================================
def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan) # Return empty inliers and a prediction function that returns NaN

    poly = PolynomialFeatures(degree=2)  # up to quadratic features
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass
        return inliers_mask, predict_func

    try:
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
        except ValueError:
            pass

    return inliers_mask, predict_func


# =======================================
# 3. Savitzky-Golay Smoothing
# =======================================
def savitzky_golay_smooth(x_val, y_val, window_length=5, polyorder=2):
    # Ensure y_val is 1D for savgol_filter
    y_flat = y_val.flatten()

    # Ensure window_length is odd and less than or equal to the number of points
    if len(y_flat) < window_length:
        window_length = len(y_flat) # Reduce window_length if data is too short
        if window_length % 2 == 0:
            window_length -= 1 # Ensure it's odd
    if window_length < polyorder + 2: # savgol_filter requires window_length > polyorder
        if len(y_flat) >= polyorder + 2:
            window_length = polyorder + 2
            if window_length % 2 == 0:
                window_length += 1
        else:
            # Not enough points for smoothing with given polyorder, return raw data
            return y_val
    if window_length < 3: # Minimum window length is 3
        return y_val
    if window_length % 2 == 0:
        window_length += 1 # Ensure it's odd

    try:
        smoothed_y = savgol_filter(y_flat, window_length, polyorder)
        return smoothed_y.reshape(-1, 1)
    except Exception as e:
        print(f"Warning: Savitzky-Golay filter failed with error: {e}. Returning original data.")
        return y_val


# =======================================
# 4. Model Selector (re-defined for self-contained execution)
# =======================================
def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None), "poly2": (-np.inf, None), "poly3": (-np.inf, None),
            "log": (-np.inf, None), "exp": (-np.inf, None), "power": (-np.inf, None),
            "root": (-np.inf, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X))

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)))

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)))

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)))
    else:
        results["log"] = (-np.inf, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
        )
    else:
        results["exp"] = (-np.inf, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X)))
        )
    else:
        results["power"] = (-np.inf, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)))
    else:
        results["root"] = (-np.inf, None)

    return results

print("Advanced helper functions (`isolation_forest_filter`, `ransac_fit`, `savitzky_golay_smooth`, `try_all_models`) defined successfully.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.ensemble import IsolationForest
from scipy.signal import savgol_filter

# The helper functions `isolation_forest_filter`, `ransac_fit`, `savitzky_golay_smooth`,
# and `try_all_models` are assumed to be already defined from a previous cell and are robust.

print("Starting comprehensive pipeline analysis for each SCALED AND CONCATENATED material data...")

# Dictionary to store all results for later visualization
analysis_results = {}

for material_name, data_dict in scaled_concatenated_data.items():
    x_initial = data_dict['x_scaled']
    y_initial = data_dict['y_concatenated']

    print(f"\nProcessing material: {material_name}")

    # --- Step 1: Isolation Forest Filtering ---
    if x_initial.shape[0] < 2:
        print(f"  Skipping Isolation Forest for '{material_name}' due to insufficient data.")
        x_if_filtered = x_initial
        y_if_filtered = y_initial
        if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)
    else:
        try:
            if_inlier_mask = isolation_forest_filter(x_initial, y_initial)
            x_if_filtered = x_initial[if_inlier_mask]
            y_if_filtered = y_initial[if_inlier_mask]
            print(f"  Isolation Forest identified {np.sum(~if_inlier_mask)} outliers, {np.sum(if_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: Isolation Forest failed for '{material_name}' with error: {e}. Proceeding with all data.")
            x_if_filtered = x_initial
            y_if_filtered = y_initial
            if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)


    # --- Step 2: RANSAC Filtering (local outlier cleaning) ---
    ransac_inlier_mask = np.ones(x_if_filtered.shape[0], dtype=bool)
    ransac_predict_func = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_if_filtered.shape[0] < 2:
        print(f"  Skipping RANSAC for '{material_name}' due to insufficient data after IF filtering.")
        x_ransac_filtered = x_if_filtered
        y_ransac_filtered = y_if_filtered
    else:
        try:
            ransac_inlier_mask, ransac_predict_func = ransac_fit(x_if_filtered, y_if_filtered)
            x_ransac_filtered = x_if_filtered[ransac_inlier_mask]
            y_ransac_filtered = y_if_filtered[ransac_inlier_mask]
            print(f"  RANSAC identified {np.sum(~ransac_inlier_mask)} local outliers, {np.sum(ransac_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: RANSAC failed for '{material_name}' with error: {e}. Proceeding with IF filtered data.")
            x_ransac_filtered = x_if_filtered
            y_ransac_filtered = y_if_filtered


    # --- Step 3: Savitzky-Golay Smoothing ---
    y_smoothed = np.full_like(y_ransac_filtered, np.nan) # Default to NaN
    if x_ransac_filtered.shape[0] < 3: # Savitzky-Golay typically needs at least 3 points
        print(f"  Skipping Savitzky-Golay for '{material_name}' due to insufficient data after RANSAC filtering.")
        y_smoothed = y_ransac_filtered # No smoothing, keep original data
    else:
        try:
            # Sort data before smoothing to ensure correct application of filter
            sort_indices = np.argsort(x_ransac_filtered.flatten())
            x_sorted_for_sg = x_ransac_filtered[sort_indices]
            y_sorted_for_sg = y_ransac_filtered[sort_indices]

            # Dynamically adjust window_length if necessary
            window_length = min(len(y_sorted_for_sg), 5) # Start with window 5
            if window_length % 2 == 0:
                window_length -= 1 # Ensure odd
            if window_length < 3: # Minimum window_length is 3
                window_length = 3

            polyorder = min(2, window_length - 1) # polyorder must be less than window_length
            if window_length <= polyorder: # Ensure window_length > polyorder
                window_length = polyorder + 1

            y_smoothed = savitzky_golay_smooth(x_sorted_for_sg, y_sorted_for_sg, window_length=window_length, polyorder=polyorder)
            # Reorder y_smoothed back to original RANSAC filtered order if data was sorted
            y_unsmoothed_reordered = np.zeros_like(y_ransac_filtered)
            y_unsmoothed_reordered[sort_indices] = y_smoothed # This will re-sort the smoothed data correctly
            y_smoothed = y_unsmoothed_reordered

            print(f"  Savitzky-Golay smoothing applied with window_length={window_length}, polyorder={polyorder}.")
        except Exception as e:
            print(f"  Warning: Savitzky-Golay smoothing failed for '{material_name}' with error: {e}. Proceeding without smoothing.")
            y_smoothed = y_ransac_filtered


    # --- Step 4: Best-fitting mathematical model ---
    best_type = None
    best_r2 = -np.inf
    best_predict = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_ransac_filtered.shape[0] < 2:
        print(f"  Skipping model fitting for '{material_name}' due to insufficient data after RANSAC filtering.")
    else:
        models = try_all_models(x_ransac_filtered, y_ransac_filtered)
        valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}
        if valid_models:
            best_type = max(valid_models, key=lambda k: valid_models[k][0])
            best_r2, best_predict = valid_models[best_type]
            print(f"  Best-fitting model: {best_type} (R²={best_r2:.3f}).")
        else:
            print(f"  No valid mathematical models could be fitted for '{material_name}'.")


    # Store all results for this material
    analysis_results[material_name] = {
        'x_initial': x_initial,
        'y_initial': y_initial,
        'if_inlier_mask': if_inlier_mask,
        'x_if_filtered': x_if_filtered,
        'y_if_filtered': y_if_filtered,
        'ransac_inlier_mask': ransac_inlier_mask,
        'x_ransac_filtered': x_ransac_filtered,
        'y_ransac_filtered': y_ransac_filtered,
        'ransac_predict_func': ransac_predict_func,
        'y_smoothed': y_smoothed,
        'best_model_type': best_type,
        'best_model_r2': best_r2,
        'best_predict_func': best_predict,
    }

print("\nComprehensive pipeline analysis complete for all materials.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `analysis_results` dictionary is assumed to be populated from the previous step

print("Starting visualization of comprehensive pipeline analysis results...")

for material_name, results in analysis_results.items():
    plt.figure(figsize=(14, 8))
    x_initial = results['x_initial']
    y_initial = results['y_initial']
    if_inlier_mask = results['if_inlier_mask']
    x_if_filtered = results['x_if_filtered']
    y_if_filtered = results['y_if_filtered']
    ransac_inlier_mask = results['ransac_inlier_mask']
    x_ransac_filtered = results['x_ransac_filtered']
    y_ransac_filtered = results['y_ransac_filtered']
    ransac_predict_func = results['ransac_predict_func']
    y_smoothed = results['y_smoothed']
    best_model_type = results['best_model_type']
    best_model_r2 = results['best_model_r2']
    best_predict_func = results['best_predict_func']

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Plot Isolation Forest filtered data (inliers)
    # Color the IF inliers differently, or only show outliers
    # For clarity, let's show the data that passed IF filter
    plt.scatter(x_initial[if_inlier_mask], y_initial[if_inlier_mask], s=30, alpha=0.7, label='После Isolation Forest (inliers)', color='skyblue')

    # Plot RANSAC filtered data (inliers) from the IF filtered data
    # We need to correctly map the RANSAC inlier mask back to the x_if_filtered data
    if len(x_ransac_filtered) > 0:
        plt.scatter(x_ransac_filtered, y_ransac_filtered, s=40, facecolors='none', edgecolors='green', linewidth=1.5, label='После RANSAC (очищенные)')

    # Plot RANSAC fitted curve (from x_if_filtered data)
    if x_if_filtered.size > 0:
        x_plot_ransac = np.linspace(x_if_filtered.min(), x_if_filtered.max(), 400).reshape(-1, 1)
        try:
            ransac_predictions = ransac_predict_func(x_plot_ransac)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot_ransac, ransac_predictions, color='orange', linewidth=2, linestyle=':', label='RANSAC модель (на IF inliers)')
        except Exception as e:
            print(f"Warning: Could not plot RANSAC curve for {material_name}: {e}")

    # Plot Savitzky-Golay smoothed curve (on RANSAC filtered data)
    if y_smoothed.size > 0 and not np.all(np.isnan(y_smoothed)) and x_ransac_filtered.size > 0:
        # Ensure x_ransac_filtered is sorted to match y_smoothed if it was sorted for SG
        sort_indices = np.argsort(x_ransac_filtered.flatten())
        x_for_sg_plot = x_ransac_filtered[sort_indices]
        y_for_sg_plot = y_smoothed[sort_indices]
        plt.plot(x_for_sg_plot, y_for_sg_plot, color='purple', linewidth=2.5, label='Savitzky-Golay (сглаженные)')

    # Plot best-fitting mathematical function (on RANSAC filtered data)
    if best_predict_func is not None and x_ransac_filtered.size > 0:
        x_plot_best_model = np.linspace(x_ransac_filtered.min(), x_ransac_filtered.max(), 400).reshape(-1, 1)
        try:
            best_model_predictions = best_predict_func(x_plot_best_model)
            if not np.all(np.isnan(best_model_predictions)):
                plt.plot(x_plot_best_model, best_model_predictions, color='red', linewidth=3, linestyle='--', label=f'Лучшая функция: {best_model_type} (R²={best_model_r2:.3f})')
        except Exception as e:
            print(f"Warning: Could not plot best model curve for {material_name}: {e}")

    plt.title(f'Анализ материала: {material_name}', fontsize=16)
    plt.xlabel('Номер попытки (масштабировано)', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Visualization complete for all materials.")


In [ ]:
import numpy as np
import pandas as pd

# 1. Extract the first column as the independent variable `attempts`
attempts = df.iloc[:, 0].values.reshape(-1, 1)

# 2. Initialize an empty dictionary to store the cleaned data for each experimental series
individual_series_data = {}

# List to keep track of identified numeric columns
identified_numeric_columns = []

# 3. Iterate through each column in the DataFrame `df`, starting from the second column
for col_name in df.columns[1:]:
    # 4. Check if its data type is numeric
    if pd.api.types.is_numeric_dtype(df[col_name]):
        identified_numeric_columns.append(col_name)

        # 5. Create a temporary DataFrame combining the `attempts` array and the current numeric column's values
        temp_series_data = pd.DataFrame({
            'attempts_val': attempts.flatten(),
            'series_val': df[col_name].values.flatten()
        })

        # 6. Drop any rows from this temporary DataFrame that contain NaN values
        cleaned_series_data = temp_series_data.dropna()

        # 7. If, after cleaning, there are at least two data points remaining
        if cleaned_series_data.shape[0] >= 2:
            x_cleaned = cleaned_series_data['attempts_val'].values.reshape(-1, 1)
            y_cleaned = cleaned_series_data['series_val'].values.reshape(-1, 1)

            # 8. Store these cleaned 2D NumPy arrays in the `individual_series_data` dictionary
            individual_series_data[col_name] = {'x': x_cleaned, 'y': y_cleaned}
        else:
            print(f"Skipping column '{col_name}' due to insufficient data after NaN removal (less than 2 data points).")

# 9. Print the names of the identified experimental columns that will be analyzed.
print("Identified experimental columns for individual analysis:")
for col_name in individual_series_data.keys():
    print(f"- {col_name} (data shape: x={individual_series_data[col_name]['x'].shape}, y={individual_series_data[col_name]['y'].shape})")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.ensemble import IsolationForest
from scipy.signal import savgol_filter

# The helper functions `isolation_forest_filter`, `ransac_fit`, `savitzky_golay_smooth`,
# and `try_all_models` are assumed to be already defined from a previous cell and are robust.
# The `individual_series_data` dictionary is assumed to be populated from the previous step.

print("Starting comprehensive pipeline analysis for each INDIVIDUAL experimental series...")

# Dictionary to store all results for later visualization
individual_series_analysis_results = {} # Ensure this is initialized

for series_name, data_dict in individual_series_data.items():
    x_initial = data_dict['x']
    y_initial = data_dict['y']

    print(f"\nProcessing series: {series_name}")

    # --- Step 1: Isolation Forest Filtering ---
    if x_initial.shape[0] < 2:
        print(f"  Skipping Isolation Forest for '{series_name}' due to insufficient data.")
        x_if_filtered = x_initial
        y_if_filtered = y_initial
        if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)
    else:
        try:
            # IF expects 2D data for x and y
            if_inlier_mask = isolation_forest_filter(x_initial, y_initial)
            x_if_filtered = x_initial[if_inlier_mask]
            y_if_filtered = y_initial[if_inlier_mask]
            print(f"  Isolation Forest identified {np.sum(~if_inlier_mask)} outliers, {np.sum(if_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: Isolation Forest failed for '{series_name}' with error: {e}. Proceeding with all data.")
            x_if_filtered = x_initial
            y_if_filtered = y_initial
            if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)


    # --- Step 2: RANSAC Filtering (local outlier cleaning) ---
    ransac_inlier_mask = np.ones(x_if_filtered.shape[0], dtype=bool)
    ransac_predict_func = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_if_filtered.shape[0] < 2:
        print(f"  Skipping RANSAC for '{series_name}' due to insufficient data after IF filtering.")
        x_ransac_filtered = x_if_filtered
        y_ransac_filtered = y_if_filtered
    else:
        try:
            ransac_inlier_mask, ransac_predict_func = ransac_fit(x_if_filtered, y_if_filtered)
            x_ransac_filtered = x_if_filtered[ransac_inlier_mask]
            y_ransac_filtered = y_if_filtered[ransac_inlier_mask]
            print(f"  RANSAC identified {np.sum(~ransac_inlier_mask)} local outliers, {np.sum(ransac_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: RANSAC failed for '{series_name}' with error: {e}. Proceeding with IF filtered data.")
            x_ransac_filtered = x_if_filtered
            y_ransac_filtered = y_if_filtered


    # --- Step 3: Savitzky-Golay Smoothing ---
    y_smoothed = np.full_like(y_ransac_filtered, np.nan) # Default to NaN
    if x_ransac_filtered.shape[0] < 3: # Savitzky-Golay typically needs at least 3 points
        print(f"  Skipping Savitzky-Golay for '{series_name}' due to insufficient data after RANSAC filtering.")
        y_smoothed = y_ransac_filtered # No smoothing, keep original data
    else:
        try:
            # Sort data before smoothing to ensure correct application of filter
            sort_indices = np.argsort(x_ransac_filtered.flatten())
            x_sorted_for_sg = x_ransac_filtered[sort_indices]
            y_sorted_for_sg = y_ransac_filtered[sort_indices]

            # Dynamically adjust window_length if necessary
            window_length = min(len(y_sorted_for_sg), 5) # Start with window 5
            if window_length % 2 == 0:
                window_length -= 1 # Ensure odd
            if window_length < 3: # Minimum window_length is 3
                window_length = 3

            polyorder = min(2, window_length - 1) # polyorder must be less than window_length
            if window_length <= polyorder: # Ensure window_length > polyorder
                window_length = polyorder + 1

            y_smoothed = savitzky_golay_smooth(x_sorted_for_sg, y_sorted_for_sg, window_length=window_length, polyorder=polyorder)
            # Reorder y_smoothed back to original RANSAC filtered order if data was sorted
            y_unsmoothed_reordered = np.zeros_like(y_ransac_filtered)
            y_unsmoothed_reordered[sort_indices] = y_smoothed # This will re-sort the smoothed data correctly
            y_smoothed = y_unsmoothed_reordered

            print(f"  Savitzky-Golay smoothing applied with window_length={window_length}, polyorder={polyorder}.")
        except Exception as e:
            print(f"  Warning: Savitzky-Golay smoothing failed for '{series_name}' with error: {e}. Proceeding without smoothing.")
            y_smoothed = y_ransac_filtered


    # --- Step 4: Best-fitting mathematical model ---
    best_type = None
    best_r2 = -np.inf
    best_predict = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_ransac_filtered.shape[0] < 2:
        print(f"  Skipping model fitting for '{series_name}' due to insufficient data after RANSAC filtering.")
    else:
        models = try_all_models(x_ransac_filtered, y_ransac_filtered)
        valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}
        if valid_models:
            best_type = max(valid_models, key=lambda k: valid_models[k][0])
            best_r2, best_predict = valid_models[best_type]
            print(f"  Best-fitting model: {best_type} (R²={best_r2:.3f}).")
        else:
            print(f"  No valid mathematical models could be fitted for '{series_name}'.")


    # Store all results for this material
    individual_series_analysis_results[series_name] = {
        'x_initial': x_initial,
        'y_initial': y_initial,
        'if_inlier_mask': if_inlier_mask,
        'x_if_filtered': x_if_filtered,
        'y_if_filtered': y_if_filtered,
        'ransac_inlier_mask': ransac_inlier_mask,
        'x_ransac_filtered': x_ransac_filtered,
        'y_ransac_filtered': y_ransac_filtered,
        'ransac_predict_func': ransac_predict_func,
        'y_smoothed': y_smoothed,
        'best_model_type': best_type,
        'best_model_r2': best_r2,
        'best_predict_func': best_predict,
    }

print("\nComprehensive pipeline analysis complete for all INDIVIDUAL experimental series.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `individual_series_analysis_results` dictionary is assumed to be populated from the previous step

print("Starting visualization of comprehensive pipeline analysis results for individual series...")

for series_name, results in individual_series_analysis_results.items():
    plt.figure(figsize=(14, 8))
    x_initial = results['x_initial']
    y_initial = results['y_initial']
    if_inlier_mask = results['if_inlier_mask']
    x_if_filtered = results['x_if_filtered']
    y_if_filtered = results['y_if_filtered']
    ransac_inlier_mask = results['ransac_inlier_mask']
    x_ransac_filtered = results['x_ransac_filtered']
    y_ransac_filtered = results['y_ransac_filtered']
    ransac_predict_func = results['ransac_predict_func']
    y_smoothed = results['y_smoothed']
    best_model_type = results['best_model_type']
    best_model_r2 = results['best_model_r2']
    best_predict_func = results['best_predict_func']

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Plot Isolation Forest filtered data (inliers)
    plt.scatter(x_initial[if_inlier_mask], y_initial[if_inlier_mask], s=30, alpha=0.7, label='После Isolation Forest (inliers)', color='skyblue')

    # Plot RANSAC filtered data (inliers) from the IF filtered data
    if len(x_ransac_filtered) > 0:
        plt.scatter(x_ransac_filtered, y_ransac_filtered, s=40, facecolors='none', edgecolors='green', linewidth=1.5, label='После RANSAC (очищенные)')

    # Plot RANSAC fitted curve (from x_if_filtered data)
    if x_if_filtered.size > 0:
        x_plot_ransac = np.linspace(x_if_filtered.min(), x_if_filtered.max(), 400).reshape(-1, 1)
        try:
            ransac_predictions = ransac_predict_func(x_plot_ransac)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot_ransac, ransac_predictions, color='orange', linewidth=2, linestyle=':', label='RANSAC модель (на IF inliers)')
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot_ransac[np.argmin(ransac_predictions)][0]
                max_x_ransac = x_plot_ransac[np.argmax(ransac_predictions)][0]
                print(f"  RANSAC Model Extrema for '{series_name}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"Warning: RANSAC model predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot RANSAC curve for {series_name}: {e}")

    # Plot Savitzky-Golay smoothed curve (on RANSAC filtered data)
    if y_smoothed.size > 0 and not np.all(np.isnan(y_smoothed)) and x_ransac_filtered.size > 0:
        # Ensure x_ransac_filtered is sorted to match y_smoothed if it was sorted for SG
        sort_indices = np.argsort(x_ransac_filtered.flatten())
        x_for_sg_plot = x_ransac_filtered[sort_indices]
        y_for_sg_plot = y_smoothed[sort_indices]
        plt.plot(x_for_sg_plot, y_for_sg_plot, color='purple', linewidth=2.5, label='Savitzky-Golay (сглаженные)')

    # Plot best-fitting mathematical function (on RANSAC filtered data)
    if best_predict_func is not None and x_ransac_filtered.size > 0:
        x_plot_best_model = np.linspace(x_ransac_filtered.min(), x_ransac_filtered.max(), 400).reshape(-1, 1)
        try:
            best_model_predictions = best_predict_func(x_plot_best_model)
            if not np.all(np.isnan(best_model_predictions)):
                model_description = f"Лучшая функция: {best_model_type} (R²={best_model_r2:.3f})"
                if best_model_type == 'linear':
                    model_description += " (Полином 1-й степени)"
                elif best_model_type == 'poly2':
                    model_description += " (Полином 2-й степени)"
                elif best_model_type == 'poly3':
                    model_description += " (Полином 3-й степени)"
                else:
                    model_description += " (Не полиномиальная функция)"

                plt.plot(x_plot_best_model, best_model_predictions, color='red', linewidth=3, linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot_best_model[np.argmin(best_model_predictions)][0]
                max_x_best = x_plot_best_model[np.argmax(best_model_predictions)][0]
                print(f"  Best Model Extrema for '{series_name}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
            else:
                print(f"Warning: Best mathematical function predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot best model curve for {series_name}: {e}")

    plt.title(f'Анализ серии: {series_name}', fontsize=16)
    plt.xlabel('Номер попытки', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Visualization complete for all individual experimental series.")

In [ ]:
import numpy as np
import pandas as pd

# The helper functions `ransac_fit` and `try_all_models` are assumed to be
# already defined from a previous cell and are robust.
# The `individual_series_data` dictionary is assumed to be populated from the previous step.

print("Starting RANSAC-only analysis for each INDIVIDUAL experimental series...")

# 1. Initialize an empty dictionary to store the analysis outcomes
ransac_only_results = {} # Ensure this is initialized

# 2. Iterate through each series_name and its corresponding data_dict
for series_name, data_dict in individual_series_data.items():
    # 3. For each series, extract the x and y data
    x_initial = data_dict['x']
    y_initial = data_dict['y']

    print(f"\nProcessing series (RANSAC-only): {series_name}")

    # --- Apply RANSAC Method ---
    ransac_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)
    ransac_predict_func = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_initial.shape[0] < 2: # RANSAC needs at least a few points
        print(f"  Skipping RANSAC for '{series_name}' due to insufficient data.")
        x_ransac_cleaned = x_initial
        y_ransac_cleaned = y_initial
    else:
        try:
            # 4. Apply the ransac_fit function
            ransac_inlier_mask, ransac_predict_func = ransac_fit(x_initial, y_initial)
            # 5. Filter the x and y data using the ransac_inlier_mask
            x_ransac_cleaned = x_initial[ransac_inlier_mask]
            y_ransac_cleaned = y_initial[ransac_inlier_mask]
            print(f"  RANSAC identified {np.sum(~ransac_inlier_mask)} outliers, {np.sum(ransac_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: RANSAC failed for '{series_name}' with error: {e}. Proceeding with original data.")
            x_ransac_cleaned = x_initial
            y_ransac_cleaned = y_initial
            ransac_inlier_mask = np.ones(x_initial.shape[0], dtype=bool) # Assume all are inliers if RANSAC fails critically

    # --- Apply try_all_models to the RANSAC cleaned data ---
    best_type = None
    best_r2 = -np.inf
    best_predict = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_ransac_cleaned.shape[0] < 2: # Model fitting needs at least 2 points
        print(f"  Skipping model fitting for '{series_name}' due to insufficient data after RANSAC filtering.")
    else:
        try:
            # 6. Apply the try_all_models function
            models = try_all_models(x_ransac_cleaned, y_ransac_cleaned)
            valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

            if valid_models:
                # 7. Identify the best_model_type, best_model_r2, and best_predict_func
                best_type = max(valid_models, key=lambda k: valid_models[k][0])
                best_r2, best_predict = valid_models[best_type]
                print(f"  Best-fitting model (RANSAC-cleaned): {best_type} (R²={best_r2:.3f}).")
            else:
                print(f"  No valid mathematical models could be fitted for '{series_name}' after RANSAC cleaning.")
        except Exception as e:
            print(f"  Warning: Model fitting failed for '{series_name}' with error: {e}.")

    # 8. Store the results for the current series
    ransac_only_results[series_name] = {
        'x_initial': x_initial,
        'y_initial': y_initial,
        'ransac_inlier_mask': ransac_inlier_mask,
        'x_ransac_cleaned': x_ransac_cleaned,
        'y_ransac_cleaned': y_ransac_cleaned,
        'ransac_predict_func': ransac_predict_func,
        'best_model_type': best_type,
        'best_model_r2': best_r2,
        'best_predict_func': best_predict,
    }

print("\nRANSAC-only analysis complete for all individual experimental series.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `ransac_only_results` and `individual_series_analysis_results` dictionaries
# are assumed to be populated from previous steps.

print("Starting visualization of comparison between RANSAC-only and Pipeline best models for individual series...")

for series_name, ransac_results in ransac_only_results.items():
    # Retrieve pipeline results for the current series
    pipeline_results = individual_series_analysis_results.get(series_name)

    if pipeline_results is None:
        print(f"Warning: No pipeline results found for series '{series_name}'. Skipping comparison plot.")
        continue

    plt.figure(figsize=(14, 8))

    # Extract initial raw data
    x_initial = ransac_results['x_initial']
    y_initial = ransac_results['y_initial']

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Define common x_plot range for evaluation
    x_plot_min = x_initial.min()
    x_plot_max = x_initial.max()

    if x_plot_min == x_plot_max:
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)

    # --- Plot Best-fitting model from RANSAC-only method ---
    best_type_ransac = ransac_results['best_model_type']
    best_r2_ransac = ransac_results['best_model_r2']
    best_predict_func_ransac = ransac_results['best_predict_func']

    if best_predict_func_ransac is not None and x_plot.size > 0 and best_r2_ransac != -np.inf:
        try:
            predictions_ransac = best_predict_func_ransac(x_plot)
            if not np.all(np.isnan(predictions_ransac)):
                model_description_ransac = f'RANSAC-only: {best_type_ransac} (R²={best_r2_ransac:.3f})'
                if best_type_ransac == 'linear':
                    model_description_ransac += " (Полином 1-й степени)"
                elif best_type_ransac == 'poly2':
                    model_description_ransac += " (Полином 2-й степени)"
                elif best_type_ransac == 'poly3':
                    model_description_ransac += " (Полином 3-й степени)"
                else:
                    model_description_ransac += " (Не полиномиальная функция)"

                plt.plot(x_plot, predictions_ransac, color='blue', linewidth=2, linestyle='-',
                         label=model_description_ransac)
                min_y_ransac_only = np.min(predictions_ransac)
                max_y_ransac_only = np.max(predictions_ransac)
                min_x_ransac_only = x_plot[np.argmin(predictions_ransac)][0]
                max_x_ransac_only = x_plot[np.argmax(predictions_ransac)][0]
                print(f"  RANSAC-only Model Extrema for '{series_name}': Min Y={min_y_ransac_only:.3f} (at X={min_x_ransac_only:.3f}), Max Y={max_y_ransac_only:.3f} (at X={max_x_ransac_only:.3f})")
            else:
                print(f"Warning: RANSAC-only predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Error plotting RANSAC-only model for '{series_name}': {e}")

    # --- Plot Best-fitting model from Pipeline method (IF+RANSAC+SG) ---
    best_type_pipeline = pipeline_results['best_model_type']
    best_r2_pipeline = pipeline_results['best_model_r2']
    best_predict_func_pipeline = pipeline_results['best_predict_func']

    if best_predict_func_pipeline is not None and x_plot.size > 0 and best_r2_pipeline != -np.inf:
        try:
            predictions_pipeline = best_predict_func_pipeline(x_plot)
            if not np.all(np.isnan(predictions_pipeline)):
                model_description_pipeline = f'Pipeline: {best_type_pipeline} (R²={best_r2_pipeline:.3f})'
                if best_type_pipeline == 'linear':
                    model_description_pipeline += " (Полином 1-й степени)"
                elif best_type_pipeline == 'poly2':
                    model_description_pipeline += " (Полином 2-й степени)"
                elif best_type_pipeline == 'poly3':
                    model_description_pipeline += " (Полином 3-й степени)"
                else:
                    model_description_pipeline += " (Не полиномиальная функция)"

                plt.plot(x_plot, predictions_pipeline, color='red', linewidth=2, linestyle='--',
                         label=model_description_pipeline)
                min_y_pipeline = np.min(predictions_pipeline)
                max_y_pipeline = np.max(predictions_pipeline)
                min_x_pipeline = x_plot[np.argmin(predictions_pipeline)][0]
                max_x_pipeline = x_plot[np.argmax(predictions_pipeline)][0]
                print(f"  Pipeline Model Extrema for '{series_name}': Min Y={min_y_pipeline:.3f} (at X={min_x_pipeline:.3f}), Max Y={max_y_pipeline:.3f} (at X={max_x_pipeline:.3f})")
            else:
                print(f"Warning: Pipeline predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Error plotting Pipeline model for '{series_name}': {e}")

    # Add plot elements
    plt.title(f'Сравнение лучших моделей для серии: {series_name}', fontsize=16)
    plt.xlabel('Номер попытки', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Comparison visualizations complete for all individual experimental series.")

# Task
To incorporate the polynomial equations into the analysis, I'll first modify the `try_all_models` function in cell `e45aaa01` to return the coefficients of the fitted polynomial models (linear, quadratic, and cubic). This will allow subsequent steps to extract these coefficients and format them into readable equation strings.

Specifically, for each polynomial model (linear, poly2, poly3), I will store the coefficients (`coef_` and `intercept_`) along with the degree of the polynomial and the `PolynomialFeatures` object (for poly2 and poly3) in the results dictionary. For other non-polynomial models, no coefficient information will be returned.

I will also update the `ransac_fit` function in the same cell to return the coefficients of its underlying quadratic polynomial model, as the plan indicates the need to extract coefficients for both RANSAC-only and Pipeline best models in comparison plots. This will involve capturing the `estimator_` of the `RANSACRegressor` and its `coef_` and `intercept_` values.

## Modify try_all_models to return coefficients

### Subtask:
Adjust the `try_all_models` function in cell `e45aaa01` to return the coefficients of the fitted polynomial models (linear, poly2, poly3) along with the R² score and prediction function. This will involve storing the `LinearRegression` model objects or their `coef_` and `intercept_` attributes.


**Reasoning**:
The user wants to modify the `try_all_models` and `ransac_fit` functions to return model coefficients. I will edit cell `e45aaa01` to update the return values for polynomial models to include `coef_`, `intercept_`, and `poly_transformer` where appropriate, and for the `ransac_fit` function to return the coefficients of the underlying linear model.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# =======================================
# 2. Набор тестируемых функций
# =======================================

def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None, None), "poly2": (-np.inf, None, None), "poly3": (-np.inf, None, None),
            "log": (-np.inf, None, None), "exp": (-np.inf, None, None), "power": (-np.inf, None, None),
            "root": (-np.inf, None, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    lin_coefs = {'coef': lin.coef_.flatten(), 'intercept': lin.intercept_}
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X), lin_coefs)

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    poly2_coefs = {'coef': model2.coef_.flatten(), 'intercept': model2.intercept_, 'poly_transformer': poly2}
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)), poly2_coefs)

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    poly3_coefs = {'coef': model3.coef_.flatten(), 'intercept': model3.intercept_, 'poly_transformer': poly3}
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)), poly3_coefs)

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)), None)
    else:
        results["log"] = (-np.inf, None, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
            None
        )
    else:
        results["exp"] = (-np.inf, None, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X))),
            None
        )
    else:
        results["power"] = (-np.inf, None, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)), None)
    else:
        results["root"] = (-np.inf, None, None)

    return results


# =======================================
# 3. Метод 1: RANSAC + Polynomial/Linear
# =======================================

def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan), None # Return empty inliers, nan predictions, and no coeffs

    poly = PolynomialFeatures(degree=2)
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN
    model_coeffs = None # Default coefficients to None

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            # Need enough samples for PolynomialFeatures to transform and then for LinearRegression
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
                model_coeffs = {'coef': lin_fallback.coef_.flatten(), 'intercept': lin_fallback.intercept_, 'poly_transformer': poly}
            else:
                # Not enough samples for even simple polyfit after transform, return NaN predictions
                predict_func = lambda X: np.full_like(X, np.nan)
                model_coeffs = None
        except ValueError:
            pass # Prediction function and coeffs remain default if fallback also fails
        return inliers_mask, predict_func, model_coeffs

    try:
        # Add random_state for reproducibility
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
        # Extract coefficients from the best estimator found by RANSAC
        model_coeffs = {'coef': ransac.estimator_.coef_.flatten(), 'intercept': ransac.estimator_.intercept_, 'poly_transformer': poly}
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        # If RANSAC fails, fall back to simple linear regression
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
                model_coeffs = {'coef': lin_fallback.coef_.flatten(), 'intercept': lin_fallback.intercept_, 'poly_transformer': poly}
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
                model_coeffs = None
        except ValueError:
            pass # Prediction function and coeffs remain default if fallback also fails

    return inliers_mask, predict_func, model_coeffs

print("Helper functions `try_all_models` and `ransac_fit` defined successfully, now returning coefficients and poly_transformer for relevant models.")

## Create a Polynomial Equation Formatter

### Subtask:
Develop a helper function that takes the coefficients of a polynomial model and its degree, then formats it into a user-friendly string representation, e.g., '3x^3 + 2x^2 + 5x + 1'. This function should handle different signs and omit terms with zero coefficients effectively.


**Reasoning**:
The subtask is to create a helper function `format_polynomial_equation` that takes polynomial coefficients and formats them into a user-friendly string. This function will be defined in a new code cell.



In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures # To handle poly_transformer if provided

def format_polynomial_equation(coefficients, intercept, poly_transformer=None, x_label='x'):
    """
    Formats polynomial coefficients into a user-friendly string representation.

    Args:
        coefficients (np.ndarray): 1D array of coefficients for powers of x (excluding intercept).
        intercept (float): The intercept term.
        poly_transformer (PolynomialFeatures, optional): The PolynomialFeatures transformer object.
                                                         If provided, used to determine feature names for degrees.
                                                         If None, degree is inferred from coefficients length.
        x_label (str): The label to use for the independent variable (e.g., 'x', 't'). Defaults to 'x'.

    Returns:
        str: A string representation of the polynomial equation.
    """
    terms = []
    # Reverse coefficients to go from highest degree to lowest
    # Note: PolynomialFeatures outputs in order x^0, x^1, x^2, ...
    # If poly_transformer is not None, coefficients will be for powers x^1, x^2, ...
    # and the intercept will be handled separately.

    if poly_transformer:
        # Coefficients from model.coef_ will correspond to poly_transformer.fit_transform output
        # which usually is [1, x, x^2, ..., x^n] for degree n. However, if the model is LinearRegression
        # fitted to poly.fit_transform(X), its .coef_ will be for x, x^2, ..., x^n (intercept handled separately)
        # We need to map these to their degrees.

        # Reconstruct the feature names to get actual powers
        feature_names = poly_transformer.get_feature_names_out([x_label])
        # Feature names are like '1', 'x', 'x^2'
        # We need to match coefficients to these degrees. Model.coef_ generally skips the '1' term if intercept is separate.
        # So, coefficients[i] corresponds to feature_names[i+1]

        # Assuming coefficients array contains terms for x^1, x^2, ..., x^n
        # and the intercept is passed separately.
        # The order of coefficients is usually x^1, x^2, ..., x^n.
        # len(coefficients) should be poly_transformer.n_output_features_ - 1 (since '1' is handled by intercept)
        if poly_transformer.include_bias: # If poly_transformer was set to include bias, but LinearRegression handled intercept separately
            # We expect coefficients to be for x^1, x^2, ...
            pass # No specific action, just a check
        else:
            # If poly_transformer explicitly excluded bias, then coefficients[0] is for x^0, [1] for x^1, etc.
            # This path is less common with separate intercept handling, but good to be robust.
            # For simplicity and common use-case where LinearRegression handles intercept, assume coefficients are for x^1, x^2, ...
            pass

        # Map coefficients to their powers based on feature_names
        # Skip the constant term from feature_names as it's the intercept
        coef_map = {}
        # coef_map: {degree: coefficient_value}

        # Need to handle if poly_transformer produces a constant feature itself
        # If poly_transformer has include_bias=True, feature_names_out will start with '1'
        # and coefficients will be for x, x^2, ..., x^degree
        # If poly_transformer has include_bias=False, feature_names_out will start with 'x'

        # Let's rebuild the degrees from feature_names
        # If feature_names[0] is '1', then coefficients array is typically for x, x^2, ...
        # If feature_names[0] is 'x', then coefficients array is for x, x^2, ...
        # The problem is that coefficients length might not directly match feature_names length minus 1
        # in all cases, e.g., if some features were dropped or custom transform.

        # A safer approach is to get the original degrees the coefficients correspond to:
        # For PolynomialFeatures, the order of coef_ is typically increasing powers of x.
        # So coef[0] for x^1, coef[1] for x^2, etc., up to coef[n-1] for x^n.
        # The `poly_transformer.powers_` attribute gives the powers of the input features
        # for each output feature. For a single input feature, it's simpler.

        # The most robust way is to just use degree from poly_transformer.degree
        # and assume coefficients are in order x^1, x^2, ..., x^degree
        current_degree = poly_transformer.degree
        # coefficients length should be `current_degree`
        if len(coefficients) != current_degree:
            # This case might happen if LinearRegression was fitted on a different set of features
            # than what poly_transformer was setup for, or if non-standard PolynomialFeatures output.
            # Fallback to inferring from coefficients length.
            max_power = len(coefficients)
        else:
            max_power = current_degree

        # Iterate from highest power down to x^1
        for i in range(max_power - 1, -1, -1):
            coef = coefficients[i]
            power = i + 1 # coefficients[0] is for x^1, coefficients[1] for x^2 etc.
            if not np.isclose(coef, 0, atol=1e-6):
                term_str = f"{coef:.3g}"
                if power == 1:
                    term_str += f"{x_label}"
                elif power > 1:
                    term_str += f"{x_label}^{power}"
                terms.append(term_str)


    else: # If no poly_transformer, infer degree from coefficients length
        # Assume coefficients are for x^n, x^(n-1), ..., x^1 (if reversed already)
        # Or, assume coefficients are for x^1, x^2, ..., x^n (if not reversed)
        # To match the order from try_all_models and ransac_fit which use LinearRegression.
        # For these, model.coef_ will be for x^1, x^2, ..., x^n (where n is the degree)
        max_power = len(coefficients)
        for i in range(max_power - 1, -1, -1):
            coef = coefficients[i]
            power = i + 1 # For coefficients[0] = x^1, coefficients[1] = x^2, ...

            if not np.isclose(coef, 0, atol=1e-6):
                term_str = f"{coef:.3g}"
                if power == 1:
                    term_str += f"{x_label}"
                elif power > 1:
                    term_str += f"{x_label}^{power}"
                terms.append(term_str)

    # Add the intercept term
    if not np.isclose(intercept, 0, atol=1e-6) or (len(terms) == 0 and np.isclose(intercept, 0, atol=1e-6)):
        # Only add if non-zero, or if it's the only term and it's zero (to avoid empty string)
        terms.append(f"{intercept:.3g}")

    # Build the final string, handling signs and empty cases
    equation = ""
    for i, term in enumerate(terms):
        if i > 0 and not term.startswith('-'):
            equation += " + "
        elif i > 0 and term.startswith('-'): # Already has a minus sign, just add space
            equation += " "
        equation += term

    # If the equation is empty (all coefficients and intercept were zero)
    if not equation.strip():
        return "0"

    return equation.strip().replace(' + -', ' - ')

print("Helper function `format_polynomial_equation` defined successfully.")

## Update Aggregated Columns Plotting

### Subtask:
Modify cell `861d0ea4` to extract polynomial coefficients from the best-fitting models, format them into an equation string using the `format_polynomial_equation` function, and display this equation in both the print output and the plot legend.


**Reasoning**:
The user wants to update cell `861d0ea4` to display polynomial equations for the best-fitting models in both the print output and the plot legend. This requires modifying the model result unpacking to include coefficients and then using the `format_polynomial_equation` function to generate the equation string.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# The variables `aggregated_data` and `columns` are assumed to be correctly defined
# from previous execution cells after the aggregation step.
# The helper functions `try_all_models` and `ransac_fit` are also assumed to be
# already defined from a previous cell and are robust.

print("Starting RANSAC and Model Fitting for each AGGREGATED experimental column...")

# Extract `attempts` from the aggregated_data DataFrame
attempts_agg = aggregated_data.iloc[:, 0].values.reshape(-1, 1)

for col in columns:
    y_raw = aggregated_data[col].values.reshape(-1, 1)

    # Combine attempts and y_raw, then drop rows with NaNs
    temp_data = pd.DataFrame({'attempts_val': attempts_agg.flatten(), 'y_val': y_raw.flatten()})
    cleaned_data = temp_data.dropna()

    # Check if there is enough data after cleaning
    if cleaned_data.shape[0] < 2: # Need at least 2 points for linear regression, more for polynomial or RANSAC
        print(f"Skipping aggregated column '{col}' due to insufficient non-NaN data after cleaning (less than 2 data points).")
        continue

    x_cleaned = cleaned_data['attempts_val'].values.reshape(-1, 1)
    y_cleaned = cleaned_data['y_val'].values.reshape(-1, 1)

    # ---- A. Метод RANSAC ----
    inliers, ransac_predict, ransac_coeffs = None, None, None
    try:
        # ransac_fit now returns coeffs
        inliers, ransac_predict, ransac_coeffs = ransac_fit(x_cleaned, y_cleaned)
    except Exception as e:
        print(f"Critical Error: ransac_fit function call failed for aggregated column '{col}'. Error: {e}. Skipping RANSAC plot for this column.")
        inliers = np.ones(len(x_cleaned), dtype=bool) # Assume all are inliers if RANSAC fails critically
        ransac_predict = lambda X: np.full_like(X, np.nan)
        ransac_coeffs = None


    # ---- B. Перебор функций ----
    models = try_all_models(x_cleaned, y_cleaned)
    # Filter models with valid prediction functions and R2 scores that are not -inf
    # try_all_models now returns coeffs, so update filtering and unpacking
    valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

    best_type = None
    best_r2 = -np.inf
    best_predict = None
    best_model_coefs = None # Initialize variable for best model coefficients

    if valid_models:
        best_type = max(valid_models, key=lambda k: valid_models[k][0])
        best_r2, best_predict, best_model_coefs = valid_models[best_type] # Unpack coefficients
    else:
        print(f"Skipping aggregated column '{col}' as no valid mathematical models could be fitted after cleaning data.")
        continue # Skip plotting if no models can be fitted


    # ---- Визуализация ----
    plt.figure(figsize=(12, 7))
    x_plot_min = x_cleaned.min()
    x_plot_max = x_cleaned.max()

    # Handle cases where x_cleaned has only one unique value (or min == max)
    if x_plot_min == x_plot_max:
        # If all x values are the same, can't create a range for plotting a curve.
        # Just plot points, no curve for continuous function.
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)


    # Plot raw data
    plt.scatter(x_cleaned, y_cleaned, s=40, label="Сырые данные", alpha=0.7, color='skyblue')

    # Plot RANSAC inliers
    if inliers is not None and len(inliers) == len(x_cleaned) and np.any(inliers):
        plt.scatter(x_cleaned[inliers], y_cleaned[inliers], s=70, label="RANSAC — без выбросов", facecolors='none', edgecolors='green', linewidth=2)
    elif inliers is not None and not np.any(inliers):
        print(f"No inliers identified by RANSAC for aggregated column '{col}'.")


    # Plot RANSAC curve
    if ransac_predict is not None and x_plot.size > 0:
        try:
            ransac_predictions = ransac_predict(x_plot)
            if not np.all(np.isnan(ransac_predictions)):
                ransac_model_description = "RANSAC модель"
                if ransac_coeffs is not None: # Format RANSAC equation if coefficients are available
                    ransac_equation_str = format_polynomial_equation(ransac_coeffs['coef'], ransac_coeffs['intercept'], ransac_coeffs['poly_transformer'], x_label='x')
                    ransac_model_description += f" ({ransac_equation_str})"

                plt.plot(x_plot, ransac_predictions, linewidth=2.5, color='orange', label=ransac_model_description)
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot[np.argmin(ransac_predictions)][0]
                max_x_ransac = x_plot[np.argmax(ransac_predictions)][0]
                print(f"  RANSAC Model Extrema for '{col}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"RANSAC model predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating RANSAC predictions for aggregated column '{col}': {e}. Not plotting RANSAC curve.")


    # Plot best mathematical function
    if best_predict is not None and x_plot.size > 0:
        try:
            best_model_predictions = best_predict(x_plot)
            if not np.all(np.isnan(best_model_predictions)): # Only plot if predictions are not all NaNs
                model_description = f"Лучшая функция: {best_type} (R²={best_r2:.3f})"

                # Format polynomial equation if coefficients are available
                if best_model_coefs is not None:
                    if 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                        poly_transformer = best_model_coefs.get('poly_transformer', None)
                        equation_str = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                        model_description += f" (y = {equation_str})"

                plt.plot(x_plot, best_model_predictions, linewidth=3, color='red', linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot[np.argmin(best_model_predictions)][0]
                max_x_best = x_plot[np.argmax(best_model_predictions)][0]
                print(f"  Best Model Extrema for '{col}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
                # Also print the equation in the output
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str_print = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    print(f"    Best Model Equation for '{col}': y = {equation_str_print}")
            else:
                print(f"Best mathematical function predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating best model predictions for aggregated column '{col}': {e}. Not plotting best model curve.")


    plt.title(f"Столбец: {col}", fontsize=16)
    plt.xlabel("Номер попытки", fontsize=12)
    plt.ylabel("Значение разности потенциалов", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("\nAnalysis and plotting complete for all identified AGGREGATED experimental columns.")

**Reasoning**:
The error message "unsupported format string passed to numpy.ndarray.__format__" indicates that a NumPy array (specifically for `intercept` and potentially `min_x_ransac`, `max_x_ransac`) is being passed to an f-string formatter that expects a scalar. This can be resolved by extracting the scalar value using `.item()` from single-element NumPy arrays. I will first apply this fix to the `try_all_models` and `ransac_fit` functions in cell `7178030a` to ensure the `intercept` values stored in the coefficient dictionaries are always scalars.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# =======================================
# 2. Набор тестируемых функций
# =======================================

def try_all_models(x, y):
    results = {}

    # Ensure x and y are not empty before fitting
    if x.size == 0 or y.size == 0:
        return {
            "linear": (-np.inf, None, None), "poly2": (-np.inf, None, None), "poly3": (-np.inf, None, None),
            "log": (-np.inf, None, None), "exp": (-np.inf, None, None), "power": (-np.inf, None, None),
            "root": (-np.inf, None, None)
        }

    # ---------- Линейная ----------
    lin = LinearRegression().fit(x, y)
    pred = lin.predict(x)
    lin_coefs = {'coef': lin.coef_.flatten(), 'intercept': lin.intercept_.item()} # Fix: .item() for scalar intercept
    results["linear"] = (r2_score(y, pred), lambda X: lin.predict(X), lin_coefs)

    # ---------- Квадратичная ----------
    poly2 = PolynomialFeatures(2)
    x2 = poly2.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred2 = model2.predict(x2)
    poly2_coefs = {'coef': model2.coef_.flatten(), 'intercept': model2.intercept_.item(), 'poly_transformer': poly2} # Fix: .item() for scalar intercept
    results["poly2"] = (r2_score(y, pred2), lambda X: model2.predict(poly2.transform(X)), poly2_coefs)

    # ---------- Кубическая ----------
    poly3 = PolynomialFeatures(3)
    x3 = poly3.fit_transform(x)
    model3 = LinearRegression().fit(x3, y)
    pred3 = model3.predict(x3)
    poly3_coefs = {'coef': model3.coef_.flatten(), 'intercept': model3.intercept_.item(), 'poly_transformer': poly3} # Fix: .item() for scalar intercept
    results["poly3"] = (r2_score(y, pred3), lambda X: model3.predict(poly3.transform(X)), poly3_coefs)

    # ---------- Логарифмическая ----------
    # Need to handle potential negative/zero values in x
    x_positive_mask = (x > 0).flatten()
    x_positive = x[x_positive_mask]
    y_for_log_x = y[x_positive_mask]

    if x_positive.size > 1: # Need at least 2 points for R2 score and fitting
        lx = np.log(x_positive).reshape(-1, 1) # Ensure 2D for LinearRegression
        y_for_log_x = y_for_log_x.reshape(-1, 1) # Ensure 2D for LinearRegression

        m = LinearRegression().fit(lx, y_for_log_x)
        results["log"] = (r2_score(y_for_log_x, m.predict(lx)), lambda X: m.predict(np.log(X)), None)
    else:
        results["log"] = (-np.inf, None, None)


    # ---------- Экспонента ----------
    # Need to handle potential negative/zero values in y
    y_positive_mask = (y > 0).flatten()
    y_positive = y[y_positive_mask]
    x_for_exp_y = x[y_positive_mask]

    if y_positive.size > 1:
        ly = np.log(y_positive).reshape(-1, 1)
        x_for_exp_y = x_for_exp_y.reshape(-1, 1)

        linexp = LinearRegression().fit(x_for_exp_y, ly)
        results["exp"] = (
            r2_score(ly, linexp.predict(x_for_exp_y)),
            lambda X: np.exp(linexp.predict(X)),
            None
        )
    else:
        results["exp"] = (-np.inf, None, None)


    # ---------- Степенная ----------
    # Need to handle potential negative/zero values in x and y
    common_positive_mask = (x > 0).flatten() & (y > 0).flatten()
    x_common_positive = x[common_positive_mask]
    y_common_positive = y[common_positive_mask]

    if x_common_positive.size > 1 and y_common_positive.size > 1:
        lx = np.log(x_common_positive).reshape(-1, 1)
        ly = np.log(y_common_positive).reshape(-1, 1)

        mp = LinearRegression().fit(lx, ly)
        results["power"] = (
            r2_score(ly, mp.predict(lx)),
            lambda X: np.exp(mp.predict(np.log(X))),
            None
        )
    else:
        results["power"] = (-np.inf, None, None)


    # ---------- Корневая ----------
    # Need to handle potential negative values in x
    x_non_negative_mask = (x >= 0).flatten()
    x_non_negative = x[x_non_negative_mask]
    y_for_root_x = y[x_non_negative_mask]

    if x_non_negative.size > 1:
        rx = np.sqrt(x_non_negative).reshape(-1, 1)
        y_for_root_x = y_for_root_x.reshape(-1, 1)

        mroot = LinearRegression().fit(rx, y_for_root_x)
        results["root"] = (r2_score(y_for_root_x, mroot.predict(rx)), lambda X: mroot.predict(np.sqrt(X)), None)
    else:
        results["root"] = (-np.inf, None, None)

    return results


# =======================================
# 3. Метод 1: RANSAC + Polynomial/Linear
# =======================================

def ransac_fit(x, y):
    # Ensure x and y are not empty
    if x.size == 0 or y.size == 0:
        return np.array([]), lambda X: np.full_like(X, np.nan), None # Return empty inliers, nan predictions, and no coeffs

    poly = PolynomialFeatures(degree=2)
    x_poly = poly.fit_transform(x)

    inliers_mask = np.ones(len(x), dtype=bool) # Default to all inliers
    predict_func = lambda X: np.full_like(X, np.nan) # Default prediction to NaN
    model_coeffs = None # Default coefficients to None

    # RANSACRegressor requires at least n_features + 1 samples (degree=2 means 3 features: 1, x, x^2 => min 4 samples)
    min_samples_for_ransac = x_poly.shape[1] + 1
    if x.shape[0] < min_samples_for_ransac:
        # Fallback to simple linear regression if RANSAC min_samples requirement is not met
        try:
            # Need enough samples for PolynomialFeatures to transform and then for LinearRegression
            if x.shape[0] >= 2: # At least 2 points for linear regression, which is base for PolyFeatures
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
                model_coeffs = {'coef': lin_fallback.coef_.flatten(), 'intercept': lin_fallback.intercept_.item(), 'poly_transformer': poly} # Fix: .item()
            else:
                # Not enough samples for even simple polyfit after transform, return NaN predictions
                predict_func = lambda X: np.full_like(X, np.nan)
                model_coeffs = None
        except ValueError:
            pass # Prediction function and coeffs remain default if fallback also fails
        return inliers_mask, predict_func, model_coeffs

    try:
        # Add random_state for reproducibility
        ransac = RANSACRegressor(random_state=42, min_samples=min_samples_for_ransac)
        ransac.fit(x_poly, y)
        inliers_mask = ransac.inlier_mask_
        predict_func = lambda X: ransac.predict(poly.transform(X))
        # Extract coefficients from the best estimator found by RANSAC
        model_coeffs = {'coef': ransac.estimator_.coef_.flatten(), 'intercept': ransac.estimator_.intercept_.item(), 'poly_transformer': poly} # Fix: .item()
    except ValueError as e:
        print(f"Warning: RANSACRegressor failed during fit: {e}. Falling back to standard linear regression.")
        # If RANSAC fails, fall back to simple linear regression
        try:
            if x.shape[0] >= 2:
                lin_fallback = LinearRegression().fit(x_poly, y)
                predict_func = lambda X: lin_fallback.predict(poly.transform(X))
                model_coeffs = {'coef': lin_fallback.coef_.flatten(), 'intercept': lin_fallback.intercept_.item(), 'poly_transformer': poly} # Fix: .item()
            else:
                predict_func = lambda X: np.full_like(X, np.nan)
                model_coeffs = None
        except ValueError:
            pass # Prediction function and coeffs remain default if fallback also fails

    return inliers_mask, predict_func, model_coeffs

print("Helper functions `try_all_models` and `ransac_fit` defined successfully, now returning coefficients and poly_transformer for relevant models.")


**Reasoning**:
The previous execution of cell `20e8c993` failed because the `format_polynomial_equation` function was not defined, as indicated by the execution results. Also, the error message `unsupported format string passed to numpy.ndarray.__format__` for extrema values (like `min_x_ransac`) indicates that these values, which are likely single-element NumPy arrays, need to be converted to scalar Python types using `.item()` before formatting. I will first re-run cell `fdb22335` to define the `format_polynomial_equation` helper function, and then I will re-execute the modified cell `20e8c993` to apply the `.item()` fix for extrema calculations and use the `format_polynomial_equation` function for polynomial models.



In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures # To handle poly_transformer if provided

def format_polynomial_equation(coefficients, intercept, poly_transformer=None, x_label='x'):
    """
    Formats polynomial coefficients into a user-friendly string representation.

    Args:
        coefficients (np.ndarray): 1D array of coefficients for powers of x (excluding intercept).
        intercept (float): The intercept term.
        poly_transformer (PolynomialFeatures, optional): The PolynomialFeatures transformer object.
                                                         If provided, used to determine feature names for degrees.
                                                         If None, degree is inferred from coefficients length.
        x_label (str): The label to use for the independent variable (e.g., 'x', 't'). Defaults to 'x'.

    Returns:
        str: A string representation of the polynomial equation.
    """
    terms = []
    # Reverse coefficients to go from highest degree to lowest
    # Note: PolynomialFeatures outputs in order x^0, x^1, x^2, ...
    # If poly_transformer is not None, coefficients will be for powers x^1, x^2, ...
    # and the intercept will be handled separately.

    if poly_transformer:
        # Coefficients from model.coef_ will correspond to poly_transformer.fit_transform output
        # which usually is [1, x, x^2, ..., x^n] for degree n. However, if the model is LinearRegression
        # fitted to poly.fit_transform(X), its .coef_ will be for x, x^2, ..., x^n (intercept handled separately)
        # We need to map these to their degrees.

        # Reconstruct the feature names to get actual powers
        feature_names = poly_transformer.get_feature_names_out([x_label])
        # Feature names are like '1', 'x', 'x^2'
        # We need to match coefficients to these degrees. Model.coef_ generally skips the '1' term if intercept is separate.
        # So, coefficients[i] corresponds to feature_names[i+1]

        # Assuming coefficients array contains terms for x^1, x^2, ..., x^n
        # and the intercept is passed separately.
        # The order of coefficients is usually x^1, x^2, ..., x^n.
        # len(coefficients) should be `poly_transformer.degree`
        current_degree = poly_transformer.degree

        # If coefficients array length does not match expected degree (e.g., if it's 0 or more than 1 term),
        # use its actual length to determine max_power.
        # Otherwise, use the degree from the polynomial transformer.
        if len(coefficients) != current_degree or current_degree == 0: # Ensure robustness for edge cases
            max_power = len(coefficients)
        else:
            max_power = current_degree

        # Iterate from highest power down to x^1
        for i in range(max_power - 1, -1, -1):
            coef = coefficients[i]
            power = i + 1 # coefficients[0] is for x^1, coefficients[1] for x^2 etc.
            if not np.isclose(coef, 0, atol=1e-6):
                term_str = f"{coef:.3g}"
                if power == 1:
                    term_str += f"{x_label}"
                elif power > 1:
                    term_str += f"{x_label}^{power}"
                terms.append(term_str)


    else: # If no poly_transformer, infer degree from coefficients length
        # Assume coefficients are for x^1, x^2, ..., x^n
        max_power = len(coefficients)
        for i in range(max_power - 1, -1, -1):
            coef = coefficients[i]
            power = i + 1 # For coefficients[0] = x^1, coefficients[1] = x^2, ...

            if not np.isclose(coef, 0, atol=1e-6):
                term_str = f"{coef:.3g}"
                if power == 1:
                    term_str += f"{x_label}"
                elif power > 1:
                    term_str += f"{x_label}^{power}"
                terms.append(term_str)

    # Add the intercept term
    if not np.isclose(intercept, 0, atol=1e-6) or (len(terms) == 0 and np.isclose(intercept, 0, atol=1e-6)): # Keep intercept if it's the only term and 0
        terms.append(f"{intercept:.3g}")

    # Build the final string, handling signs and empty cases
    equation = ""
    for i, term in enumerate(terms):
        if i > 0 and not term.startswith('-'):
            equation += " + "
        elif i > 0 and term.startswith('-'): # Already has a minus sign, just add space
            equation += " "
        equation += term

    # If the equation is empty (all coefficients and intercept were zero)
    if not equation.strip():
        return "0"

    return equation.strip().replace(' + -', ' - ')

print("Helper function `format_polynomial_equation` redefined successfully.")


**Reasoning**:
The `format_polynomial_equation` helper function has been redefined and the `try_all_models` and `ransac_fit` functions have been updated to return scalar intercepts. Now I will re-execute cell `20e8c993` to apply the fixes for displaying the polynomial equations and for the extrema value formatting. Specifically, I will add `.item()` to the `min_x_ransac`, `max_x_ransac`, `min_x_best`, and `max_x_best` assignments.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

# The variables `aggregated_data` and `columns` are assumed to be correctly defined
# from previous execution cells after the aggregation step.
# The helper functions `try_all_models` and `ransac_fit` are also assumed to be
# already defined from a previous cell and are robust.

print("Starting RANSAC and Model Fitting for each AGGREGATED experimental column...")

# Extract `attempts` from the aggregated_data DataFrame
attempts_agg = aggregated_data.iloc[:, 0].values.reshape(-1, 1)

for col in columns:
    y_raw = aggregated_data[col].values.reshape(-1, 1)

    # Combine attempts and y_raw, then drop rows with NaNs
    temp_data = pd.DataFrame({'attempts_val': attempts_agg.flatten(), 'y_val': y_raw.flatten()})
    cleaned_data = temp_data.dropna()

    # Check if there is enough data after cleaning
    if cleaned_data.shape[0] < 2: # Need at least 2 points for linear regression, more for polynomial or RANSAC
        print(f"Skipping aggregated column '{col}' due to insufficient non-NaN data after cleaning (less than 2 data points).")
        continue

    x_cleaned = cleaned_data['attempts_val'].values.reshape(-1, 1)
    y_cleaned = cleaned_data['y_val'].values.reshape(-1, 1)

    # ---- A. Метод RANSAC ----
    inliers, ransac_predict, ransac_coeffs = None, None, None
    try:
        # ransac_fit now returns coeffs
        inliers, ransac_predict, ransac_coeffs = ransac_fit(x_cleaned, y_cleaned)
    except Exception as e:
        print(f"Critical Error: ransac_fit function call failed for aggregated column '{col}'. Error: {e}. Skipping RANSAC plot for this column.")
        inliers = np.ones(len(x_cleaned), dtype=bool) # Assume all are inliers if RANSAC fails critically
        ransac_predict = lambda X: np.full_like(X, np.nan)
        ransac_coeffs = None


    # ---- B. Перебор функций ----
    models = try_all_models(x_cleaned, y_cleaned)
    # Filter models with valid prediction functions and R2 scores that are not -inf
    # try_all_models now returns coeffs, so update filtering and unpacking
    valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

    best_type = None
    best_r2 = -np.inf
    best_predict = None
    best_model_coefs = None # Initialize variable for best model coefficients

    if valid_models:
        best_type = max(valid_models, key=lambda k: valid_models[k][0])
        best_r2, best_predict, best_model_coefs = valid_models[best_type] # Unpack coefficients
    else:
        print(f"Skipping aggregated column '{col}' as no valid mathematical models could be fitted after cleaning data.")
        continue # Skip plotting if no models can be fitted


    # ---- Визуализация ----
    plt.figure(figsize=(12, 7))
    x_plot_min = x_cleaned.min()
    x_plot_max = x_cleaned.max()

    # Handle cases where x_cleaned has only one unique value (or min == max)
    if x_plot_min == x_plot_max:
        # If all x values are the same, can't create a range for plotting a curve.
        # Just plot points, no curve for continuous function.
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)


    # Plot raw data
    plt.scatter(x_cleaned, y_cleaned, s=40, label="Сырые данные", alpha=0.7, color='skyblue')

    # Plot RANSAC inliers
    if inliers is not None and len(inliers) == len(x_cleaned) and np.any(inliers):
        plt.scatter(x_cleaned[inliers], y_cleaned[inliers], s=70, label="RANSAC — без выбросов", facecolors='none', edgecolors='green', linewidth=2)
    elif inliers is not None and not np.any(inliers):
        print(f"No inliers identified by RANSAC for aggregated column '{col}'.")


    # Plot RANSAC curve
    if ransac_predict is not None and x_plot.size > 0:
        try:
            ransac_predictions = ransac_predict(x_plot)
            if not np.all(np.isnan(ransac_predictions)):
                ransac_model_description = "RANSAC модель"
                if ransac_coeffs is not None: # Format RANSAC equation if coefficients are available
                    ransac_equation_str = format_polynomial_equation(ransac_coeffs['coef'], ransac_coeffs['intercept'], ransac_coeffs['poly_transformer'], x_label='x')
                    ransac_model_description += f" (y = {ransac_equation_str})"

                plt.plot(x_plot, ransac_predictions, linewidth=2.5, color='orange', label=ransac_model_description)
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot[np.argmin(ransac_predictions)][0].item() # Fix: .item()
                max_x_ransac = x_plot[np.argmax(ransac_predictions)][0].item() # Fix: .item()
                print(f"  RANSAC Model Extrema for '{col}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"RANSAC model predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating RANSAC predictions for aggregated column '{col}': {e}. Not plotting RANSAC curve.")


    # Plot best mathematical function
    if best_predict is not None and x_plot.size > 0:
        try:
            best_model_predictions = best_predict(x_plot)
            if not np.all(np.isnan(best_model_predictions)): # Only plot if predictions are not all NaNs
                model_description = f"Лучшая функция: {best_type} (R²={best_r2:.3f})"

                # Format polynomial equation if coefficients are available
                if best_model_coefs is not None:
                    if 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                        poly_transformer = best_model_coefs.get('poly_transformer', None)
                        equation_str = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                        model_description += f" (y = {equation_str})"

                plt.plot(x_plot, best_model_predictions, linewidth=3, color='red', linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot[np.argmin(best_model_predictions)][0].item() # Fix: .item()
                max_x_best = x_plot[np.argmax(best_model_predictions)][0].item() # Fix: .item()
                print(f"  Best Model Extrema for '{col}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
                # Also print the equation in the output
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str_print = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    print(f"    Best Model Equation for '{col}': y = {equation_str_print}")
            else:
                print(f"Best mathematical function predictions for column '{col}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Error generating best model predictions for aggregated column '{col}': {e}. Not plotting best model curve.")


    plt.title(f"Столбец: {col}", fontsize=16)
    plt.xlabel("Номер попытки", fontsize=12)
    plt.ylabel("Значение разности потенциалов", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("\nAnalysis and plotting complete for all identified AGGREGATED experimental columns.")

## Update Individual Series Plotting

### Subtask:
Modify cell `e177975a` to use the updated `try_all_models` function. When the best-fitting model is a polynomial, extract its coefficients, use the new formatter function to get the equation string, and print this equation string along with the existing extrema information.


**Reasoning**:
The subtask requires modifying cell `e178975a` to incorporate the coefficients and formatted equation strings for polynomial models, and to ensure correct scalar formatting for extrema values. I will update the cell to unpack `best_model_coefs`, use the `format_polynomial_equation` function to generate the equation string for the legend and print output, and apply `.item()` to the extrema values.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `individual_series_analysis_results` dictionary is assumed to be populated from the previous step

print("Starting visualization of comprehensive pipeline analysis results for individual series...")

for series_name, results in individual_series_analysis_results.items():
    plt.figure(figsize=(14, 8))
    x_initial = results['x_initial']
    y_initial = results['y_initial']
    if_inlier_mask = results['if_inlier_mask']
    x_if_filtered = results['x_if_filtered']
    y_if_filtered = results['y_if_filtered']
    # RANSAC results from pipeline
    ransac_inlier_mask = results['ransac_inlier_mask']
    x_ransac_filtered = results['x_ransac_filtered']
    y_ransac_filtered = results['y_ransac_filtered']
    ransac_predict_func = results['ransac_predict_func']
    y_smoothed = results['y_smoothed']
    best_model_type = results['best_model_type']
    best_model_r2 = results['best_model_r2']
    best_predict_func = results['best_predict_func']
    best_model_coefs = results.get('best_model_coefs', None) # Unpack coefficients

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Plot Isolation Forest filtered data (inliers)
    plt.scatter(x_initial[if_inlier_mask], y_initial[if_inlier_mask], s=30, alpha=0.7, label='После Isolation Forest (inliers)', color='skyblue')

    # Plot RANSAC filtered data (inliers) from the IF filtered data
    if len(x_ransac_filtered) > 0:
        plt.scatter(x_ransac_filtered, y_ransac_filtered, s=40, facecolors='none', edgecolors='green', linewidth=1.5, label='После RANSAC (очищенные)')

    # Plot RANSAC fitted curve (from x_if_filtered data)
    if x_if_filtered.size > 0:
        x_plot_ransac = np.linspace(x_if_filtered.min(), x_if_filtered.max(), 400).reshape(-1, 1)
        try:
            ransac_predictions = ransac_predict_func(x_plot_ransac)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot_ransac, ransac_predictions, color='orange', linewidth=2, linestyle=':', label='RANSAC модель (на IF inliers)')
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot_ransac[np.argmin(ransac_predictions)][0].item() # Fix: .item()
                max_x_ransac = x_plot_ransac[np.argmax(ransac_predictions)][0].item() # Fix: .item()
                print(f"  RANSAC Model Extrema for '{series_name}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"Warning: RANSAC model predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot RANSAC curve for {series_name}: {e}")

    # Plot Savitzky-Golay smoothed curve (on RANSAC filtered data)
    if y_smoothed.size > 0 and not np.all(np.isnan(y_smoothed)) and x_ransac_filtered.size > 0:
        # Ensure x_ransac_filtered is sorted to match y_smoothed if it was sorted for SG
        sort_indices = np.argsort(x_ransac_filtered.flatten())
        x_for_sg_plot = x_ransac_filtered[sort_indices]
        y_for_sg_plot = y_smoothed[sort_indices]
        plt.plot(x_for_sg_plot, y_for_sg_plot, color='purple', linewidth=2.5, label='Savitzky-Golay (сглаженные)')

    # Plot best-fitting mathematical function (on RANSAC filtered data)
    if best_predict_func is not None and x_ransac_filtered.size > 0:
        x_plot_best_model = np.linspace(x_ransac_filtered.min(), x_ransac_filtered.max(), 400).reshape(-1, 1)
        try:
            best_model_predictions = best_predict_func(x_plot_best_model)
            if not np.all(np.isnan(best_model_predictions)):
                model_description = f"Лучшая функция: {best_model_type} (R²={best_model_r2:.3f})"

                # Format polynomial equation if coefficients are available
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    model_description += f" (y = {equation_str})"

                plt.plot(x_plot_best_model, best_model_predictions, color='red', linewidth=3, linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot_best_model[np.argmin(best_model_predictions)][0].item() # Fix: .item()
                max_x_best = x_plot_best_model[np.argmax(best_model_predictions)][0].item() # Fix: .item()
                print(f"  Best Model Extrema for '{series_name}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
                # Also print the equation in the output
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str_print = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    print(f"    Best Model Equation for '{series_name}': y = {equation_str_print}")
            else:
                print(f"Warning: Best mathematical function predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot best model curve for {series_name}: {e}")

    plt.title(f'Анализ серии: {series_name}', fontsize=16)
    plt.xlabel('Номер попытки', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Visualization complete for all individual experimental series.")

**Reasoning**:
The current subtask requires displaying polynomial equations, which depend on `best_model_coefs` being stored in the `individual_series_analysis_results` dictionary. Reviewing the notebook state, cell `4898778e` is responsible for populating this dictionary, but it currently does not save `best_model_coefs`. Therefore, before proceeding with modifications to `e178975a`, I need to update cell `4898778e` to store `best_model_coefs` for each series.

**Reasoning**:
I need to modify cell `4898778e` to ensure that `best_model_coefs` is stored in the `individual_series_analysis_results` dictionary. This will allow subsequent cells (like `e178975a`) to access and use these coefficients for equation formatting and display.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.ensemble import IsolationForest
from scipy.signal import savgol_filter

# The helper functions `isolation_forest_filter`, `ransac_fit`, `savitzky_golay_smooth`,
# and `try_all_models` are assumed to be already defined from a previous cell and are robust.
# The `individual_series_data` dictionary is assumed to be populated from the previous step.

print("Starting comprehensive pipeline analysis for each INDIVIDUAL experimental series...")

# Dictionary to store all results for later visualization
individual_series_analysis_results = {} # Ensure this is initialized

for series_name, data_dict in individual_series_data.items():
    x_initial = data_dict['x']
    y_initial = data_dict['y']

    print(f"\nProcessing series: {series_name}")

    # --- Step 1: Isolation Forest Filtering ---
    if x_initial.shape[0] < 2:
        print(f"  Skipping Isolation Forest for '{series_name}' due to insufficient data.")
        x_if_filtered = x_initial
        y_if_filtered = y_initial
        if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)
    else:
        try:
            # IF expects 2D data for x and y
            if_inlier_mask = isolation_forest_filter(x_initial, y_initial)
            x_if_filtered = x_initial[if_inlier_mask]
            y_if_filtered = y_initial[if_inlier_mask]
            print(f"  Isolation Forest identified {np.sum(~if_inlier_mask)} outliers, {np.sum(if_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: Isolation Forest failed for '{series_name}' with error: {e}. Proceeding with all data.")
            x_if_filtered = x_initial
            y_if_filtered = y_initial
            if_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)


    # --- Step 2: RANSAC Filtering (local outlier cleaning) ---
    ransac_inlier_mask = np.ones(x_if_filtered.shape[0], dtype=bool)
    ransac_predict_func = lambda X: np.full_like(X, np.nan) # Default to NaN

    if x_if_filtered.shape[0] < 2:
        print(f"  Skipping RANSAC for '{series_name}' due to insufficient data after IF filtering.")
        x_ransac_filtered = x_if_filtered
        y_ransac_filtered = y_if_filtered
    else:
        try:
            # ransac_fit now returns coefficients as well
            ransac_inlier_mask, ransac_predict_func, ransac_coeffs_for_pipeline = ransac_fit(x_if_filtered, y_if_filtered)
            x_ransac_filtered = x_if_filtered[ransac_inlier_mask]
            y_ransac_filtered = y_if_filtered[ransac_inlier_mask]
            print(f"  RANSAC identified {np.sum(~ransac_inlier_mask)} local outliers, {np.sum(ransac_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: RANSAC failed for '{series_name}' with error: {e}. Proceeding with IF filtered data.")
            x_ransac_filtered = x_if_filtered
            y_ransac_filtered = y_if_filtered
            ransac_coeffs_for_pipeline = None # Reset if RANSAC fails


    # --- Step 3: Savitzky-Golay Smoothing ---
    y_smoothed = np.full_like(y_ransac_filtered, np.nan) # Default to NaN
    if x_ransac_filtered.shape[0] < 3: # Savitzky-Golay typically needs at least 3 points
        print(f"  Skipping Savitzky-Golay for '{series_name}' due to insufficient data after RANSAC filtering.")
        y_smoothed = y_ransac_filtered # No smoothing, keep original data
    else:
        try:
            # Sort data before smoothing to ensure correct application of filter
            sort_indices = np.argsort(x_ransac_filtered.flatten())
            x_sorted_for_sg = x_ransac_filtered[sort_indices]
            y_sorted_for_sg = y_ransac_filtered[sort_indices]

            # Dynamically adjust window_length if necessary
            window_length = min(len(y_sorted_for_sg), 5) # Start with window 5
            if window_length % 2 == 0:
                window_length -= 1 # Ensure odd
            if window_length < 3: # Minimum window_length is 3
                window_length = 3

            polyorder = min(2, window_length - 1) # polyorder must be less than window_length
            if window_length <= polyorder: # Ensure window_length > polyorder
                window_length = polyorder + 1

            y_smoothed = savitzky_golay_smooth(x_sorted_for_sg, y_sorted_for_sg, window_length=window_length, polyorder=polyorder)
            # Reorder y_smoothed back to original RANSAC filtered order if data was sorted
            y_unsmoothed_reordered = np.zeros_like(y_ransac_filtered)
            # Ensure `sort_indices` are valid for assigning to `y_unsmoothed_reordered`
            if len(y_unsmoothed_reordered) == len(y_smoothed):
                y_unsmoothed_reordered[sort_indices] = y_smoothed # This will re-sort the smoothed data correctly
                y_smoothed = y_unsmoothed_reordered
            else: # Fallback if sizes don't match after sorting/smoothing
                y_smoothed = y_ransac_filtered

            print(f"  Savitzky-Golay smoothing applied with window_length={window_length}, polyorder={polyorder}.")
        except Exception as e:
            print(f"  Warning: Savitzky-Golay smoothing failed for '{series_name}' with error: {e}. Proceeding without smoothing.")
            y_smoothed = y_ransac_filtered


    # --- Step 4: Best-fitting mathematical model ---
    best_type = None
    best_r2 = -np.inf
    best_predict = lambda X: np.full_like(X, np.nan) # Default to NaN
    best_model_coefs = None # Initialize variable for best model coefficients

    if x_ransac_filtered.shape[0] < 2:
        print(f"  Skipping model fitting for '{series_name}' due to insufficient data after RANSAC filtering.")
    else:
        # try_all_models now returns coefficients as well
        models = try_all_models(x_ransac_filtered, y_ransac_filtered)
        # Filter models with valid prediction functions and R2 scores that are not -inf
        valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}
        if valid_models:
            best_type = max(valid_models, key=lambda k: valid_models[k][0])
            best_r2, best_predict, best_model_coefs = valid_models[best_type] # Unpack coefficients
            print(f"  Best-fitting model: {best_type} (R²={best_r2:.3f}).")
        else:
            print(f"  No valid mathematical models could be fitted for '{series_name}'.")


    # Store all results for this material
    individual_series_analysis_results[series_name] = {
        'x_initial': x_initial,
        'y_initial': y_initial,
        'if_inlier_mask': if_inlier_mask,
        'x_if_filtered': x_if_filtered,
        'y_if_filtered': y_if_filtered,
        'ransac_inlier_mask': ransac_inlier_mask,
        'x_ransac_filtered': x_ransac_filtered,
        'y_ransac_filtered': y_ransac_filtered,
        'ransac_predict_func': ransac_predict_func,
        'ransac_coeffs': ransac_coeffs_for_pipeline, # Store RANSAC coeffs
        'y_smoothed': y_smoothed,
        'best_model_type': best_type,
        'best_model_r2': best_r2,
        'best_predict_func': best_predict,
        'best_model_coefs': best_model_coefs, # Store best model coeffs
    }

print("\nComprehensive pipeline analysis complete for all INDIVIDUAL experimental series.")


**Reasoning**:
Now that `best_model_coefs` is correctly populated in `individual_series_analysis_results` (due to the modification of cell `4898778e`), I will re-execute cell `e178975a` with the necessary changes to unpack these coefficients, use the `format_polynomial_equation` helper function for display in the legend, and print the equation along with extrema values, ensuring all extrema are formatted as scalars using `.item()`.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `individual_series_analysis_results` dictionary is assumed to be populated from the previous step

print("Starting visualization of comprehensive pipeline analysis results for individual series...")

for series_name, results in individual_series_analysis_results.items():
    plt.figure(figsize=(14, 8))
    x_initial = results['x_initial']
    y_initial = results['y_initial']
    if_inlier_mask = results['if_inlier_mask']
    x_if_filtered = results['x_if_filtered']
    y_if_filtered = results['y_if_filtered']
    # RANSAC results from pipeline
    ransac_inlier_mask = results['ransac_inlier_mask']
    x_ransac_filtered = results['x_ransac_filtered']
    y_ransac_filtered = results['y_ransac_filtered']
    ransac_predict_func = results['ransac_predict_func']
    y_smoothed = results['y_smoothed']
    best_model_type = results['best_model_type']
    best_model_r2 = results['best_model_r2']
    best_predict_func = results['best_predict_func']
    best_model_coefs = results.get('best_model_coefs', None) # Unpack coefficients, safely handle if not present

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Plot Isolation Forest filtered data (inliers)
    plt.scatter(x_initial[if_inlier_mask], y_initial[if_inlier_mask], s=30, alpha=0.7, label='После Isolation Forest (inliers)', color='skyblue')

    # Plot RANSAC filtered data (inliers) from the IF filtered data
    if len(x_ransac_filtered) > 0:
        plt.scatter(x_ransac_filtered, y_ransac_filtered, s=40, facecolors='none', edgecolors='green', linewidth=1.5, label='После RANSAC (очищенные)')

    # Plot RANSAC fitted curve (from x_if_filtered data)
    if x_if_filtered.size > 0:
        x_plot_ransac = np.linspace(x_if_filtered.min(), x_if_filtered.max(), 400).reshape(-1, 1)
        try:
            ransac_predictions = ransac_predict_func(x_plot_ransac)
            if not np.all(np.isnan(ransac_predictions)):
                plt.plot(x_plot_ransac, ransac_predictions, color='orange', linewidth=2, linestyle=':', label='RANSAC модель (на IF inliers)')
                min_y_ransac = np.min(ransac_predictions)
                max_y_ransac = np.max(ransac_predictions)
                min_x_ransac = x_plot_ransac[np.argmin(ransac_predictions)][0].item() # Fix: .item()
                max_x_ransac = x_plot_ransac[np.argmax(ransac_predictions)][0].item() # Fix: .item()
                print(f"  RANSAC Model Extrema for '{series_name}': Min Y={min_y_ransac:.3f} (at X={min_x_ransac:.3f}), Max Y={max_y_ransac:.3f} (at X={max_x_ransac:.3f})")
            else:
                print(f"Warning: RANSAC model predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot RANSAC curve for {series_name}: {e}")

    # Plot Savitzky-Golay smoothed curve (on RANSAC filtered data)
    if y_smoothed.size > 0 and not np.all(np.isnan(y_smoothed)) and x_ransac_filtered.size > 0:
        # Ensure x_ransac_filtered is sorted to match y_smoothed if it was sorted for SG
        sort_indices = np.argsort(x_ransac_filtered.flatten())
        x_for_sg_plot = x_ransac_filtered[sort_indices]
        y_for_sg_plot = y_smoothed[sort_indices]
        plt.plot(x_for_sg_plot, y_for_sg_plot, color='purple', linewidth=2.5, label='Savitzky-Golay (сглаженные)')

    # Plot best-fitting mathematical function (on RANSAC filtered data)
    if best_predict_func is not None and x_ransac_filtered.size > 0:
        x_plot_best_model = np.linspace(x_ransac_filtered.min(), x_ransac_filtered.max(), 400).reshape(-1, 1)
        try:
            best_model_predictions = best_predict_func(x_plot_best_model)
            if not np.all(np.isnan(best_model_predictions)):
                model_description = f"Лучшая функция: {best_model_type} (R²={best_model_r2:.3f})"

                # Format polynomial equation if coefficients are available
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    model_description += f" (y = {equation_str})"

                plt.plot(x_plot_best_model, best_model_predictions, color='red', linewidth=3, linestyle='--', label=model_description)
                min_y_best = np.min(best_model_predictions)
                max_y_best = np.max(best_model_predictions)
                min_x_best = x_plot_best_model[np.argmin(best_model_predictions)][0].item() # Fix: .item()
                max_x_best = x_plot_best_model[np.argmax(best_model_predictions)][0].item() # Fix: .item()
                print(f"  Best Model Extrema for '{series_name}': Min Y={min_y_best:.3f} (at X={min_x_best:.3f}), Max Y={max_y_best:.3f} (at X={max_x_best:.3f})")
                # Also print the equation in the output
                if best_model_coefs is not None and 'coef' in best_model_coefs and 'intercept' in best_model_coefs:
                    poly_transformer = best_model_coefs.get('poly_transformer', None)
                    equation_str_print = format_polynomial_equation(best_model_coefs['coef'], best_model_coefs['intercept'], poly_transformer, x_label='x')
                    print(f"    Best Model Equation for '{series_name}': y = {equation_str_print}")
            else:
                print(f"Warning: Best mathematical function predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Could not plot best model curve for {series_name}: {e}")

    plt.title(f'Анализ серии: {series_name}', fontsize=16)
    plt.xlabel('Номер попытки', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Visualization complete for all individual experimental series.")


## Update RANSAC-only Analysis to Store Coefficients

### Subtask:
Modify cell `e5fa5610` to store the `best_model_coefs` within the `ransac_only_results` dictionary. This will ensure that coefficients are available for displaying polynomial equations in the comparison plot.


**Reasoning**:
The subtask requires modifying cell `e5fa5610` to store `best_model_coefs` within the `ransac_only_results` dictionary. I will update the cell's code to include this information in the stored results.



In [ ]:
import numpy as np
import pandas as pd

# The helper functions `ransac_fit` and `try_all_models` are assumed to be
# already defined from a previous cell and are robust.
# The `individual_series_data` dictionary is assumed to be populated from the previous step.

print("Starting RANSAC-only analysis for each INDIVIDUAL experimental series...")

# 1. Initialize an empty dictionary to store the analysis outcomes
ransac_only_results = {} # Ensure this is initialized

# 2. Iterate through each series_name and its corresponding data_dict
for series_name, data_dict in individual_series_data.items():
    # 3. For each series, extract the x and y data
    x_initial = data_dict['x']
    y_initial = data_dict['y']

    print(f"\nProcessing series (RANSAC-only): {series_name}")

    # --- Apply RANSAC Method ---
    ransac_inlier_mask = np.ones(x_initial.shape[0], dtype=bool)
    ransac_predict_func = lambda X: np.full_like(X, np.nan) # Default to NaN
    ransac_coeffs_for_ransac_only = None # Initialize RANSAC coeffs for this scope

    if x_initial.shape[0] < 2: # RANSAC needs at least a few points
        print(f"  Skipping RANSAC for '{series_name}' due to insufficient data.")
        x_ransac_cleaned = x_initial
        y_ransac_cleaned = y_initial
    else:
        try:
            # 4. Apply the ransac_fit function
            # ransac_fit now returns coefficients as well
            ransac_inlier_mask, ransac_predict_func, ransac_coeffs_for_ransac_only = ransac_fit(x_initial, y_initial)
            # 5. Filter the x and y data using the ransac_inlier_mask
            x_ransac_cleaned = x_initial[ransac_inlier_mask]
            y_ransac_cleaned = y_initial[ransac_inlier_mask]
            print(f"  RANSAC identified {np.sum(~ransac_inlier_mask)} outliers, {np.sum(ransac_inlier_mask)} inliers.")
        except Exception as e:
            print(f"  Warning: RANSAC failed for '{series_name}' with error: {e}. Proceeding with original data.")
            x_ransac_cleaned = x_initial
            y_ransac_cleaned = y_initial
            ransac_inlier_mask = np.ones(x_initial.shape[0], dtype=bool) # Assume all are inliers if RANSAC fails critically
            ransac_coeffs_for_ransac_only = None # Reset if RANSAC fails

    # --- Apply try_all_models to the RANSAC cleaned data ---
    best_type = None
    best_r2 = -np.inf
    best_predict = lambda X: np.full_like(X, np.nan) # Default to NaN
    best_model_coefs = None # Initialize variable for best model coefficients

    if x_ransac_cleaned.shape[0] < 2: # Model fitting needs at least 2 points
        print(f"  Skipping model fitting for '{series_name}' due to insufficient data after RANSAC filtering.")
    else:
        try:
            # 6. Apply the try_all_models function
            # try_all_models now returns coefficients as well
            models = try_all_models(x_ransac_cleaned, y_ransac_cleaned)
            valid_models = {k: v for k, v in models.items() if v[1] is not None and v[0] != -np.inf}

            if valid_models:
                # 7. Identify the best_model_type, best_model_r2, and best_predict_func
                best_type = max(valid_models, key=lambda k: valid_models[k][0])
                best_r2, best_predict, best_model_coefs = valid_models[best_type] # Unpack coefficients
                print(f"  Best-fitting model (RANSAC-cleaned): {best_type} (R²={best_r2:.3f}).")
            else:
                print(f"  No valid mathematical models could be fitted for '{series_name}' after RANSAC cleaning.")
        except Exception as e:
            print(f"  Warning: Model fitting failed for '{series_name}' with error: {e}.")

    # 8. Store the results for the current series
    ransac_only_results[series_name] = {
        'x_initial': x_initial,
        'y_initial': y_initial,
        'ransac_inlier_mask': ransac_inlier_mask,
        'x_ransac_cleaned': x_ransac_cleaned,
        'y_ransac_cleaned': y_ransac_cleaned,
        'ransac_predict_func': ransac_predict_func,
        'ransac_coeffs': ransac_coeffs_for_ransac_only, # Store RANSAC coeffs (from the initial RANSAC step)
        'best_model_type': best_type,
        'best_model_r2': best_r2,
        'best_predict_func': best_predict,
        'best_model_coefs': best_model_coefs, # Store best model coeffs
    }

print("\nRANSAC-only analysis complete for all individual experimental series.")


**Reasoning**:
The subtask is to update cell `b6c62758` to display polynomial equations for both RANSAC-only and Pipeline best models, utilizing the `best_model_coefs` now stored in `ransac_only_results` and `individual_series_analysis_results` respectively. I will also ensure extrema values are converted to scalars using `.item()` to prevent formatting errors.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# `ransac_only_results` and `individual_series_analysis_results` dictionaries
# are assumed to be populated from previous steps.

print("Starting visualization of comparison between RANSAC-only and Pipeline best models for individual series...")

for series_name, ransac_results in ransac_only_results.items():
    # Retrieve pipeline results for the current series
    pipeline_results = individual_series_analysis_results.get(series_name)

    if pipeline_results is None:
        print(f"Warning: No pipeline results found for series '{series_name}'. Skipping comparison plot.")
        continue

    plt.figure(figsize=(14, 8))

    # Extract initial raw data
    x_initial = ransac_results['x_initial']
    y_initial = ransac_results['y_initial']

    # Plot initial raw data
    plt.scatter(x_initial, y_initial, s=20, alpha=0.5, label='Исходные данные (сырые)', color='grey')

    # Define common x_plot range for evaluation
    x_plot_min = x_initial.min()
    x_plot_max = x_initial.max()

    if x_plot_min == x_plot_max:
        x_plot = np.array([x_plot_min]).reshape(-1, 1)
    else:
        x_plot = np.linspace(x_plot_min, x_plot_max, 400).reshape(-1, 1)

    # --- Plot Best-fitting model from RANSAC-only method ---
    best_type_ransac = ransac_results['best_model_type']
    best_r2_ransac = ransac_results['best_model_r2']
    best_predict_func_ransac = ransac_results['best_predict_func']
    best_model_coefs_ransac = ransac_results.get('best_model_coefs', None)

    if best_predict_func_ransac is not None and x_plot.size > 0 and best_r2_ransac != -np.inf:
        try:
            predictions_ransac = best_predict_func_ransac(x_plot)
            if not np.all(np.isnan(predictions_ransac)):
                model_description_ransac = f'RANSAC-only: {best_type_ransac} (R²={best_r2_ransac:.3f})'

                # Format polynomial equation if coefficients are available
                if best_model_coefs_ransac is not None and 'coef' in best_model_coefs_ransac and 'intercept' in best_model_coefs_ransac:
                    poly_transformer_ransac = best_model_coefs_ransac.get('poly_transformer', None)
                    equation_str_ransac = format_polynomial_equation(best_model_coefs_ransac['coef'], best_model_coefs_ransac['intercept'], poly_transformer_ransac, x_label='x')
                    model_description_ransac += f" (y = {equation_str_ransac})"
                else:
                    model_description_ransac += " (Не полиномиальная функция)"

                plt.plot(x_plot, predictions_ransac, color='blue', linewidth=2, linestyle='-', label=model_description_ransac)
                min_y_ransac_only = np.min(predictions_ransac)
                max_y_ransac_only = np.max(predictions_ransac)
                min_x_ransac_only = x_plot[np.argmin(predictions_ransac)][0].item()
                max_x_ransac_only = x_plot[np.argmax(predictions_ransac)][0].item()
                print(f"  RANSAC-only Model Extrema for '{series_name}': Min Y={min_y_ransac_only:.3f} (at X={min_x_ransac_only:.3f}), Max Y={max_y_ransac_only:.3f} (at X={max_x_ransac_only:.3f})")
                # Also print the equation in the output
                if best_model_coefs_ransac is not None and 'coef' in best_model_coefs_ransac and 'intercept' in best_model_coefs_ransac:
                    poly_transformer_ransac_print = best_model_coefs_ransac.get('poly_transformer', None)
                    equation_str_ransac_print = format_polynomial_equation(best_model_coefs_ransac['coef'], best_model_coefs_ransac['intercept'], poly_transformer_ransac_print, x_label='x')
                    print(f"    RANSAC-only Best Model Equation for '{series_name}': y = {equation_str_ransac_print}")
            else:
                print(f"Warning: RANSAC-only predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Error plotting RANSAC-only model for '{series_name}': {e}")

    # --- Plot Best-fitting model from Pipeline method (IF+RANSAC+SG) ---
    best_type_pipeline = pipeline_results['best_model_type']
    best_r2_pipeline = pipeline_results['best_model_r2']
    best_predict_func_pipeline = pipeline_results['best_predict_func']
    best_model_coefs_pipeline = pipeline_results.get('best_model_coefs', None)

    if best_predict_func_pipeline is not None and x_plot.size > 0 and best_r2_pipeline != -np.inf:
        try:
            predictions_pipeline = best_predict_func_pipeline(x_plot)
            if not np.all(np.isnan(predictions_pipeline)):
                model_description_pipeline = f'Pipeline: {best_type_pipeline} (R²={best_r2_pipeline:.3f})'

                # Format polynomial equation if coefficients are available
                if best_model_coefs_pipeline is not None and 'coef' in best_model_coefs_pipeline and 'intercept' in best_model_coefs_pipeline:
                    poly_transformer_pipeline = best_model_coefs_pipeline.get('poly_transformer', None)
                    equation_str_pipeline = format_polynomial_equation(best_model_coefs_pipeline['coef'], best_model_coefs_pipeline['intercept'], poly_transformer_pipeline, x_label='x')
                    model_description_pipeline += f" (y = {equation_str_pipeline})"
                else:
                    model_description_pipeline += " (Не полиномиальная функция)"

                plt.plot(x_plot, predictions_pipeline, color='red', linewidth=2, linestyle='--', label=model_description_pipeline)
                min_y_pipeline = np.min(predictions_pipeline)
                max_y_pipeline = np.max(predictions_pipeline)
                min_x_pipeline = x_plot[np.argmin(predictions_pipeline)][0].item()
                max_x_pipeline = x_plot[np.argmax(predictions_pipeline)][0].item()
                print(f"  Pipeline Model Extrema for '{series_name}': Min Y={min_y_pipeline:.3f} (at X={min_x_pipeline:.3f}), Max Y={max_y_pipeline:.3f} (at X={max_x_pipeline:.3f})")
                # Also print the equation in the output
                if best_model_coefs_pipeline is not None and 'coef' in best_model_coefs_pipeline and 'intercept' in best_model_coefs_pipeline:
                    poly_transformer_pipeline_print = best_model_coefs_pipeline.get('poly_transformer', None)
                    equation_str_pipeline_print = format_polynomial_equation(best_model_coefs_pipeline['coef'], best_model_coefs_pipeline['intercept'], poly_transformer_pipeline_print, x_label='x')
                    print(f"    Pipeline Best Model Equation for '{series_name}': y = {equation_str_pipeline_print}")
            else:
                print(f"Warning: Pipeline predictions for '{series_name}' resulted in NaNs, not plotting.")
        except Exception as e:
            print(f"Warning: Error plotting Pipeline model for '{series_name}': {e}")

    # Add plot elements
    plt.title(f'Сравнение лучших моделей для серии: {series_name}', fontsize=16)
    plt.xlabel('Номер попытки', fontsize=12)
    plt.ylabel('Значение', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

print("Comparison visualizations complete for all individual experimental series.")


## Final Task

### Subtask:
Confirm that all plots and print outputs now include the explicit polynomial equations for the best-fitting models as requested, and provide a summary of the analysis.


## Summary:

### Q&A
Yes, all relevant plots and print outputs now include the explicit polynomial equations for the best-fitting models as requested.

### Data Analysis Key Findings
*   **Coefficient Extraction**: The `try_all_models` and `ransac_fit` functions were successfully updated to return the coefficients and `PolynomialFeatures` transformer objects for linear, quadratic, and cubic models. Non-polynomial models return `None` for coefficients.
*   **Polynomial Equation Formatting**: A new helper function, `format_polynomial_equation`, was created to convert model coefficients and intercepts into a user-friendly string representation (e.g., "$y = 3.45x^2 + 1.23x - 0.5$"), handling signs, zero coefficients, and different powers.
*   **Plot and Output Enhancement (Aggregated Data)**: The plotting mechanism for aggregated columns was updated to display the formatted polynomial equations in both the plot legends and console printouts for the RANSAC and best-fitting models.
*   **Plot and Output Enhancement (Individual Series Data)**:
    *   An upstream code cell (`4898778e`) was modified to correctly store polynomial coefficients for both RANSAC and best-fitting models within the `individual_series_analysis_results` dictionary, resolving a dependency.
    *   The individual series plotting code (`e177975a`) was then updated to retrieve and display these polynomial equations in plot legends and console outputs.
*   **Plot and Output Enhancement (RANSAC-only Comparison)**: The RANSAC-only analysis (`e5fa5610`) was updated to store polynomial coefficients, and the comparison plotting code (`b6c62758`) was enhanced to display these equations in both the legend and print output, for instance: "RANSAC-only Best Model Equation for 'Хлопок (серия 1)': $y = 9.49e-06x^4 -0.00268x^3 + 0.272x^2 + 4.27$".
*   **Robustness Improvements**: Throughout the process, issues related to `TypeError` (e.g., passing single-element NumPy arrays to string formatters expecting scalars) were addressed by explicitly converting these values using `.item()`, making the output formatting more robust.

### Insights or Next Steps
*   The explicit polynomial equations provide a quantitative and easily interpretable summary of the observed trends, allowing for direct comparison of model complexity and fit across different series and aggregation levels.
*   Future analysis could leverage these explicit equations for further mathematical manipulation, such as calculating derivatives to determine rates of change or extrema points analytically, providing deeper insights into the material behavior.
